In [105]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [106]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import openpyxl
import datetime
import re
import string
import unicodedata
import requests
import requests
import pytz

from bs4 import BeautifulSoup
from datetime import datetime as dt
from openpyxl import load_workbook
from pathlib import Path



from date_extractor import extract_dates


# Automate Login

In [107]:
# Automate login and scraping

# the URL of the login page
login_url = "https://www.tilastopaja.info/login.php"

# the payload with your login credentials
payload = {
    "user": "SAA",
    "password": "Alexisthe12.",
}

# send the POST request to login
response = requests.post(login_url, data=payload)


# if the request went Ok, you should get a 200 status
print(f"Status code: {response.status_code}")

# parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# find the page title
page_title = soup.title.string

# print the result page title
print(f"Page title: {page_title}")

Status code: 200
Page title: Tilastopaja Login


# Open Directory Files and Process

In [108]:
# LATEST VERSION
# Enhanced to extract specific date, and wind from triple jump and long jump events

def scraper_new(soup, file):

    import re
    import datetime
    import pandas as pd

    matches = []
    lists = []
    counts = []
    values = []
    names = []
    wind = ''
    stage = ''
    event = ''
    sub_event = ''

    gender = ''
    heat = ''
    division = ''
    dict_result = []
    dict_result2 = []
    row = []

    current_event_date = ''
    default_meet_date = ''
    page_date_range = ''

    temp_df = pd.DataFrame(columns=[
        'RANK', 'NAME', 'NATIONALITY', 'RESULT', 'QUALIFICATION',
        'COMPETITION', 'YEAR', 'DATE', 'EVENT', 'VENUE', 'GENDER',
        'STAGE', 'HEAT', 'WIND', 'DOB', 'DIVISION', 'REMARKS',
        'RX_TIME', 'DICT_RESULTS', 'SOURCE', 'REGION', 'HOST_CITY',
        'SUB_EVENT'
    ])

    def clean_cell_text(x):
        return str(x).replace('\xa0', ' ').strip()

    def looks_like_day_month(text):
        return bool(re.fullmatch(r'\d{1,2}\s+[A-Za-z]+', clean_cell_text(text)))

    def build_full_event_date(day_month, year):
        """
        Converts:
          '24 April' + '2026' -> '2026-04-24'
        """
        try:
            dt = datetime.datetime.strptime(f"{clean_cell_text(day_month)} {year}", "%d %B %Y")
            return dt.strftime("%Y-%m-%d")
        except Exception:
            return clean_cell_text(day_month)


    def parse_title_dates(title_text):
        """
        Returns:
          year, default_meet_date, page_date_range
    
        Examples:
          '... - 3 May 2026'
              -> ('2026', '2026-05-03', '3 May 2026')
    
          '... - 12 April 2026 Race walk'
              -> ('2026', '2026-04-12', '12 April 2026')
    
          '... - 24 April - 5 May 2026'
              -> ('2026', '', '24 April - 5 May 2026')
        """
        title_text = clean_cell_text(title_text)
    
        # Check range first
        m_range = re.search(
            r'-\s*(\d{1,2}\s+[A-Za-z]+)\s*-\s*(\d{1,2}\s+[A-Za-z]+\s+\d{4})\b',
            title_text
        )
        if m_range:
            raw_range = f"{m_range.group(1)} - {m_range.group(2)}"
            year_match = re.search(r'(\d{4})$', m_range.group(2))
            year = year_match.group(1) if year_match else ''
            return year, '', raw_range
    
        # Then single-date title, allowing extra trailing words like "Race walk"
        m_single = re.search(
            r'-\s*(\d{1,2}\s+[A-Za-z]+\s+\d{4})\b',
            title_text
        )
        if m_single:
            raw_date = m_single.group(1)
            dt = datetime.datetime.strptime(raw_date, "%d %B %Y")
            return dt.strftime("%Y"), dt.strftime("%Y-%m-%d"), raw_date
    
        year_match = re.search(r'20\d{2}', title_text)
        year = year_match.group(0) if year_match else ''
        return year, '', ''
    
    for div in soup.find_all('div', attrs={"class": "left"}):

        info = div.find('h1')
        text = info.text

        title_parts = text.split(',')
        print('title_parts', title_parts)

        # HOST CITY
        host_city = title_parts[1].strip() if len(title_parts) > 1 else ''

        # COMPETITION
        result = title_parts[0].split(':')
        competition = result[1].strip() if len(result) > 1 else title_parts[0].strip()

        # Parse title-level dates
        year, default_meet_date, page_date_range = parse_title_dates(text)
        current_event_date = default_meet_date if default_meet_date else page_date_range

        # VENUE
        venue_text = div.find('h2')
        venue = venue_text.text if venue_text else ''

    for i in soup.find_all('table'):

        for element in i.find_all('tr'):

            counts = []
            values = []

            for count, value in enumerate(element.find_all('td')):

                for n in value.find_all('a'):
                    if len(n.text) > 5 and '(' not in n.text and '1y' not in n.text and '/' not in n.text:
                        names.append(n.text)

                for m in value.find_all('b'):

                    if 'Multievents' in m.text:
                        sub_event = 'Multievents'

                    if 'U20' in m.text or 'U18' in m.text or 'U17' in m.text or 'U16' in m.text or 'U15' in m.text or 'U14' in m.text:
                        division = m.text
                    else:
                        heat = m.text

                    dict_result = ''

                if len(lists) == 2:

                    wind = ''
                    dict_result = ''
                    dict_result2 = ''

                    if '4 x 100m' not in event and '4 x 400m' not in event:
                        heat = ''
                        stage = ''

                    if 'Men' in lists[1] and len(lists[1]) == 3:
                        gender = 'Male'

                    if 'Women' in lists[1]:
                        gender = 'Female'

                if 'Wind' in value.text:
                    string = value.text
                    wind = string[5:]

                counts.append(count)
                values.append(value.text)

            lists = dict(zip(counts, values))

            # ---------------------------------
            # Round row with wind, e.g. Heat 1 + Wind
            # ---------------------------------
            if len(lists) == 3 and 'Wind' in clean_cell_text(lists[2]):
                heat = clean_cell_text(lists[1])
                wind = clean_cell_text(lists[2])

                if 'Wind' in clean_cell_text(lists[2]):
                    wind = clean_cell_text(lists[2])[5:].strip()

                dict_result = []

            # ---------------------------------
            # Round/stage row without wind
            # e.g. Heats / Semifinals and sometimes a date in col 3
            # ---------------------------------
            if len(lists) == 3 and 'Wind' not in clean_cell_text(lists[2]):

                cell1 = clean_cell_text(lists[1])
                cell2 = clean_cell_text(lists[2])

                if 'Semifinals' in cell1 or 'Semifinal' in cell1:
                    stage = 'Semifinals'

                if 'Heats' in cell1:
                    stage = 'Heats'

                if 'Final' in cell1:
                    stage = 'Final'

                if 'Qualification' in cell1:
                    stage = 'Qualification'

                if 'Heat 1' in cell1:
                    heat = 'Heat 1'

                if 'Heat 2' in cell1:
                    heat = 'Heat 2'

                if 'Heat 3' in cell1:
                    heat = 'Heat 3'

                if 'Heat 4' in cell1:
                    heat = 'Heat 4'

                if 'Heat 5' in cell1:
                    heat = 'Heat 5'

                if 'Race 1' in cell1:
                    heat = 'Race 1'

                if 'Race 2' in cell1:
                    heat = 'Race 2'

                if 'Race 3' in cell1:
                    heat = 'Race 3'

                # Multi-day meet: round row carries actual date
                if looks_like_day_month(cell2):
                    current_event_date = build_full_event_date(cell2, year)
                # Single-day meet: keep default meet date if round row is blank
                elif default_meet_date and not current_event_date:
                    current_event_date = default_meet_date

                dict_result = []

            # ---------------------------------
            # Event header row
            # e.g. Event | 26 April | Wind: ...
            # ---------------------------------
            if len(lists) == 5:
                event = clean_cell_text(lists[1])
                wind = ' '
                stage = ' '
                heat = ' '

                dict_result = []
                dict_result2 = []

                event_date_text = clean_cell_text(lists[2])

                # Multi-day meet: event row date present
                if looks_like_day_month(event_date_text):
                    current_event_date = build_full_event_date(event_date_text, year)
                # Single-day meet: event date cell blank, use meet date
                elif default_meet_date:
                    current_event_date = default_meet_date

                if 'Wind' in clean_cell_text(lists[4]):
                    wind = clean_cell_text(lists[4])[5:].strip()
                    print('WINDWIND', wind)

            if len(lists) == 11:
                dict_result = lists

            if len(lists) == 2 and 'Men' not in lists[1]:
                dict_result2 = lists

            if len(lists) == 2 and 'Women' not in lists[1]:
                dict_result2 = lists

            print(
                'EVENT', event,
                'GENDER', gender,
                'STAGE', stage,
                'HEAT', heat,
                'WIND', wind,
                'DATE', current_event_date,
                'DIVISION', division,
                'DICT_RESULT', dict_result,
                'ATTEMPTS', dict_result2
            )

            athlete_wind = wind

            if isinstance(dict_result, dict) and len(dict_result) > 6 and ('Long Jump' in event or 'Triple Jump' in event):
                candidate_wind = str(dict_result[6]).strip()
                if candidate_wind and candidate_wind != '\xa0':
                    athlete_wind = candidate_wind

            try:

                if 'High Jump' in event or 'Pole Vault' in event or 'Long Jump' in event or 'Triple Jump' in event or 'Shot Put' in event or 'Discus Throw' in event or 'Hammer Throw' in event or 'Javelin Throw' in event or 'Decathlon' in event or '4 x 100m' in event or '4 x 400m' in event or 'Heptathlon' in event:

                    if len(dict_result2) > 0:

                        row.append(dict_result[0])

                        if '4 x 100m' not in event and '4 x 400m' not in event and 'Mixed Relay' not in event:
                            row.append(names[-1])
                        else:
                            row.append(' ')

                        remarks = dict_result[8] + ' ' + dict_result[2]

                        if 'PB' in remarks:
                            remarks = 'PB'
                        elif 'SB' in remarks:
                            remarks = 'SB'

                        row.append(dict_result[3])
                        row.append(dict_result[5])
                        row.append(dict_result[9])
                        row.append(competition)
                        row.append(year)
                        row.append(current_event_date)
                        row.append(event)
                        row.append(venue[7:] if len(venue) > 7 else venue)
                        row.append(gender)
                        row.append(stage)
                        row.append(heat)
                        row.append(athlete_wind)
                        row.append(dict_result[4])
                        row.append(division)
                        row.append(remarks)
                        row.append(dict_result[10])
                        row.append(dict_result2)
                        row.append(file)
                        row.append('International')
                        row.append(host_city)
                        row.append(sub_event)

                    elif len(dict_result2) == 0:

                        row.append(dict_result[0])

                        if '4 x 100m' not in event and '4 x 400m' not in event and 'Mixed Relay' not in event:
                            row.append(names[-1])
                        else:
                            row.append(' ')

                        remarks = dict_result[8] + ' ' + dict_result[2]

                        if 'PB' in remarks:
                            remarks = 'PB'
                        elif 'SB' in remarks:
                            remarks = 'SB'

                        row.append(dict_result[3])
                        row.append(dict_result[5])
                        row.append(dict_result[9])
                        row.append(competition)
                        row.append(year)
                        row.append(current_event_date)
                        row.append(event)
                        row.append(venue[7:] if len(venue) > 7 else venue)
                        row.append(gender)
                        row.append(stage)
                        row.append(heat)
                        row.append(athlete_wind)
                        row.append(dict_result[4])
                        row.append(division)
                        row.append(remarks)
                        row.append(dict_result[10])
                        row.append(dict_result2)
                        row.append(file)
                        row.append('International')
                        row.append(host_city)
                        row.append(sub_event)

                elif (event and dict_result[3] and dict_result[5]) is not None:

                    row.append(dict_result[0])

                    if '4 x 100m' not in event and '4 x 400m' not in event and 'Mixed Relay' not in event:
                        row.append(names[-1])
                    else:
                        row.append(' ')

                    remarks = dict_result[8] + ' ' + dict_result[2]

                    if 'PB' in remarks:
                        remarks = 'PB'
                    elif 'SB' in remarks:
                        remarks = 'SB'

                    row.append(dict_result[3])
                    row.append(dict_result[5])
                    row.append(dict_result[9])
                    row.append(competition)
                    row.append(year)
                    row.append(current_event_date)
                    row.append(event)
                    row.append(venue[7:] if len(venue) > 7 else venue)
                    row.append(gender)
                    row.append(stage)
                    row.append(heat)
                    row.append(athlete_wind)
                    row.append(dict_result[4])
                    row.append(division)
                    row.append(remarks)
                    row.append(dict_result[10])
                    row.append(dict_result2)
                    row.append(file)
                    row.append('International')
                    row.append(host_city)
                    row.append(sub_event)

            except Exception as e:
                print("SCRAPER ERROR:", e)
                print("EVENT:", event)
                print("DICT_RESULT:", dict_result)
                print("CURRENT ROW LENGTH:", len(row))
                row = []

            if len(row) == len(temp_df.columns):
                temp_df.loc[len(temp_df)] = row
            elif len(row) > 0:
                print("ROW LENGTH MISMATCH:", len(row), "EXPECTED:", len(temp_df.columns))
                print("ROW:", row)

            dict_result2 = []
            row = []

        wind = ''
        heat = ''
        lists = []
        dict_result = []
        row = []

    return temp_df

In [109]:
# LATEST VERSON
# Test open all files in directory

#os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Asian Athletics Championship/')
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug/batch2/')


def sort_directory(directory):
    items = os.listdir(directory)
    sorted_items = sorted(items)
    return sorted_items



directory = r"/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug/batch2/"
#directory = r"/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Asian Athletics Championship/"
 
    
    
# Iterate over files in directory

sorted_items = sort_directory(directory)

files = sorted_items

print('number', len(sorted_items))

files_clean = [ file for file in files if file.endswith(".html") ]

#sorted_items.pop(0)  # remove first item (DS_store)

master_df=pd.DataFrame()  # initialize empty master df



for file in files_clean:
    
    print(file)
    
    HTMLFileToBeOpened = open(file, "r") 

    contents = HTMLFileToBeOpened.read() 
    soup = BeautifulSoup(contents, 'lxml') 
    
    temp_df=scraper_new(soup, file)    
    
    
    master_df=pd.concat([master_df, temp_df], axis=0)
    
# Reset validation state whenever a new file is loaded

SOURCE_SCHEMA_VALIDATED = False
DOMAIN_VALIDATED = False
DUPLICATE_UPLOAD_VALIDATED = False
PREUPLOAD_VALIDATED = False
POST_UPLOAD_RECONCILED = False


number 9
Tilastopaja - 26th World Masters Athletics Championships - Aug 26.html
title_parts ['RESULTS: 26th World Masters Athletics Championships', ' Daegu', ' KOR  - 22 August - 3 September 2026 Race walk']
EVENT  GENDER  STAGE  HEAT  WIND  DATE 22 August - 3 September 2026 DIVISION  DICT_RESULT [] ATTEMPTS []
SCRAPER ERROR: list index out of range
EVENT: 
DICT_RESULT: []
CURRENT ROW LENGTH: 0
EVENT  GENDER  STAGE  HEAT Multievents WIND  DATE 22 August - 3 September 2026 DIVISION  DICT_RESULT  ATTEMPTS {0: '\xa0', 1: 'Multievents'}
SCRAPER ERROR: string index out of range
EVENT: 
DICT_RESULT: 
CURRENT ROW LENGTH: 0
EVENT  GENDER  STAGE  HEAT  WIND  DATE 22 August - 3 September 2026 DIVISION  DICT_RESULT  ATTEMPTS {0: '\xa0', 1: 'Men'}
SCRAPER ERROR: string index out of range
EVENT: 
DICT_RESULT: 
CURRENT ROW LENGTH: 0
WINDWIND -1.0
EVENT 100m GENDER Male STAGE   HEAT   WIND -1.0 DATE 2026-08-22 DIVISION  DICT_RESULT [] ATTEMPTS []
SCRAPER ERROR: list index out of range
EVENT: 100m
DIC

EVENT Marathon     GENDER Male STAGE   HEAT   WIND   DATE  5 - 6 July  DIVISION  DICT_RESULT {0: '34', 1: "Samuel\xa0TollSamuel\xa0Toll\xa0('98)", 2: '', 3: 'AUS', 4: '28 Jan 98', 5: '2:25:30 ', 6: '', 7: '\xa0', 8: 'PB »', 9: '', 10: ' '} ATTEMPTS []
EVENT Marathon     GENDER Male STAGE   HEAT   WIND   DATE  5 - 6 July  DIVISION  DICT_RESULT {0: '35', 1: "Nur\xa0ShodiqNur\xa0Shodiq\xa0('92)", 2: '', 3: 'INA', 4: '17 May 92', 5: '2:26:09 ', 6: '', 7: '\xa0', 8: 'PB »', 9: '', 10: ' '} ATTEMPTS []
EVENT Marathon     GENDER Male STAGE   HEAT   WIND   DATE  5 - 6 July  DIVISION  DICT_RESULT {0: '36', 1: "Jordan\xa0SkellyJordan\xa0Skelly\xa0('92)", 2: '', 3: 'GBR', 4: '31 Jul 92', 5: '2:26:32 ', 6: '', 7: '\xa0', 8: 'SB «', 9: '', 10: ' '} ATTEMPTS []
EVENT Marathon     GENDER Male STAGE   HEAT   WIND   DATE  5 - 6 July  DIVISION  DICT_RESULT {0: '37', 1: "Jacob\xa0FosterJacob\xa0Foster\xa0('91)", 2: '', 3: 'AUS', 4: '18 Nov 91', 5: '2:26:47 ', 6: '', 7: '\xa0', 8: 'PB »', 9: '', 10: ' '} 

HOST CITY  
VENUE events: 
EVENT  GENDER  STAGE  HEAT  WIND  DATE  5 July  DIVISION  DICT_RESULT [] ATTEMPTS []
EVENT  GENDER  STAGE  HEAT Men WIND  DATE  5 July  DIVISION  DICT_RESULT  ATTEMPTS {0: '\xa0', 1: 'Men'}
EVENT Pole Vault     GENDER Male STAGE   HEAT   WIND   DATE  5 July  DIVISION  DICT_RESULT [] ATTEMPTS []
EVENT Pole Vault     GENDER Male STAGE   HEAT   WIND   DATE  5 July  DIVISION  DICT_RESULT {0: '1', 1: "Koen\xa0van der WijstKoen\xa0van der Wijst\xa0('97)", 2: '', 3: 'NED', 4: '7 Aug 97', 5: '5.44 ', 6: '', 7: '\xa0', 8: 'SB «', 9: '', 10: ' '} ATTEMPTS []
EVENT Pole Vault     GENDER Male STAGE   HEAT   WIND   DATE  5 July  DIVISION  DICT_RESULT {0: '1', 1: "Koen\xa0van der WijstKoen\xa0van der Wijst\xa0('97)", 2: '', 3: 'NED', 4: '7 Aug 97', 5: '5.44 ', 6: '', 7: '\xa0', 8: 'SB «', 9: '', 10: ' '} ATTEMPTS {0: '\xa0', 1: '5.10/1      5.30/1      5.44/1      5.51/XXX      '}
EVENT Pole Vault     GENDER Male STAGE  HEAT  WIND  DATE  5 July  DIVISION  DICT_RESULT {0: '

In [110]:
SOURCE_REQUIRED_COLUMNS = {
    'NAME',
    'RESULT',
    'EVENT',
    'GENDER',
    'COMPETITION',
    'DATE',
}

missing_source_columns = (
    SOURCE_REQUIRED_COLUMNS
    - set(master_df.columns)
)

if missing_source_columns:
    raise ValueError(
        'Input file missing required columns: '
        f'{sorted(missing_source_columns)}'
    )

SOURCE_COLUMNS_ORIGINAL = list(master_df.columns)

SOURCE_SCHEMA_VALIDATED = True

In [111]:
master_df

,RANK,NAME,NATIONALITY,RESULT,QUALIFICATION,COMPETITION,YEAR,DATE,EVENT,VENUE,...,WIND,DOB,DIVISION,REMARKS,RX_TIME,DICT_RESULTS,SOURCE,REGION,HOST_CITY,SUB_EVENT
0,1,Ward Hazen,CAN,13.58,,26th World Masters Athletics Championships,2026,2026-08-22,100m,events:,...,-1.0,28 Jul 54,,SB,,[],Tilastopaja - 26th World Masters Athletics Cha...,International,Daegu,Multievents
1,1,John MacDermott,IRL,16.32,,26th World Masters Athletics Championships,2026,2026-08-22,100m,events:,...,-1.0,12 Mar 44,,SB,,[],Tilastopaja - 26th World Masters Athletics Cha...,International,Daegu,Multievents
2,1,Robert Bonenberg,CAN,16.82,,26th World Masters Athletics Championships,2026,2026-08-22,100m,events:,...,-1.0,7 Feb 50,,PB,,[],Tilastopaja - 26th World Masters Athletics Cha...,International,Daegu,Multievents
3,1,Manuel Clemente Rocha,ARG,19.30,,26th World Masters Athletics Championships,2026,2026-08-22,100m,events:,...,-1.0,9 Mar 41,,PB,,[],Tilastopaja - 26th World Masters Athletics Cha...,International,Daegu,Multievents
4,2,Russell Jacquet-Acea,USA,15.08,,26th World Masters Athletics Championships,2026,2026-08-22,100m,events:,...,-1.0,12 Nov 52,,SB,,[],Tilastopaja - 26th World Masters Athletics Cha...,International,Daegu,Multievents
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,2,Iroha Taniyama,JPN,52.86,,The 21st Twilight Games,2026,2026-08-08,Javelin Throw,Okura Sports Park Athletics Field,...,,26 Sep 05,,PB,,[],Tilastopaja - The 21st Twilight Games - Aug 26...,International,Tokyo,
191,3,Miyabi Sono,JPN,51.41,,The 21st Twilight Games,2026,2026-08-08,Javelin Throw,Okura Sports Park Athletics Field,...,,9 Apr 05,,« »,,[],Tilastopaja - The 21st Twilight Games - Aug 26...,International,Tokyo,
192,4,Moka Nakamura,JPN,47.92,,The 21st Twilight Games,2026,2026-08-08,Javelin Throw,Okura Sports Park Athletics Field,...,,8 Nov 06,,PB,,[],Tilastopaja - The 21st Twilight Games - Aug 26...,International,Tokyo,
193,5,Hina Tsutsumi,JPN,46.97,,The 21st Twilight Games,2026,2026-08-08,Javelin Throw,Okura Sports Park Athletics Field,...,,7 Jan 05,,« »,,[],Tilastopaja - The 21st Twilight Games - Aug 26...,International,Tokyo,


In [112]:
# Quick high level check

master_df[['COMPETITION', 'EVENT', 'STAGE', 'HEAT', 'DATE']].drop_duplicates().sort_values(['DATE', 'COMPETITION', 'EVENT', 'STAGE', 'HEAT'])

,COMPETITION,EVENT,STAGE,HEAT,DATE
0,The 21st Twilight Games,100m,,Race 1,2026-08-08
7,The 21st Twilight Games,100m,,Race 2,2026-08-08
143,The 21st Twilight Games,100m Hurdles,,,2026-08-08
144,The 21st Twilight Games,100m Hurdles,,Race 1,2026-08-08
151,The 21st Twilight Games,100m Hurdles,,Race 2,2026-08-08
...,...,...,...,...,...
236,9. International Wiesław Maniak Memorial,High Jump,,,2026-08-26
252,9. International Wiesław Maniak Memorial,Javelin Throw,,,2026-08-26
250,9. International Wiesław Maniak Memorial,Javelin Throw,,,2026-08-26
101,9. International Wiesław Maniak Memorial,Pole Vault,,,2026-08-26


In [113]:
# Remove special characters
    
for col in master_df.columns:
    master_df[col] = master_df[col].astype(str)
    master_df[col] = master_df[col].str.replace('\xa0', ' ', regex=True)
    master_df[col] = master_df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    master_df[col] = master_df[col].str.replace('\r', ' ', regex=True)
    master_df[col] = master_df[col].str.replace('\n', ' ', regex=True)
    master_df[col] = master_df[col].str.strip()


    


In [114]:
# removes duplicate rows from those with list of attempts

master_df = master_df.drop_duplicates(['RANK', 'NAME', 'NATIONALITY', 'RESULT', 'QUALIFICATION', 'COMPETITION', 'YEAR', 'DATE', 'EVENT', 'VENUE', 'GENDER', 'STAGE', 'HEAT', 'DOB', 'REMARKS', 'RX_TIME', 'HOST_CITY'], keep='last')

In [115]:
# removes duplicate rows from those with list of attempts for relays

master_df = master_df.drop_duplicates(['RANK', 'NAME', 'NATIONALITY', 'RESULT', 'QUALIFICATION', 'COMPETITION', 'YEAR', 'DATE', 'EVENT', 'VENUE', 'GENDER', 'STAGE', 'HEAT', 'WIND', 'DOB', 'REMARKS', 'RX_TIME', 'HOST_CITY'], keep='last')

In [116]:
# Map general event category 

master_df.loc[master_df['EVENT'].str.contains(r'^100m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'^400m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'^600m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'^60m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'^200m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'50m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'80m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'300m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
master_df.loc[master_df['EVENT'].str.contains(r'^3000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
master_df.loc[master_df['EVENT'].str.contains(r'Mile', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'^5000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
master_df.loc[master_df['EVENT'].str.contains(r'^2000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'^1000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'^800m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'^1500m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
master_df.loc[master_df['EVENT'].str.contains(r'10,000m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
master_df.loc[master_df['EVENT'].str.contains(r'5km', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
master_df.loc[master_df['EVENT'].str.contains(r'10km', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
master_df.loc[master_df['EVENT'].str.contains(r'Cross Country', na=True), 'CATEGORY_EVENT'] = 'Cross Country'


# Override values

master_df.loc[master_df['EVENT'].str.contains(r'Discus', na=True), 'CATEGORY_EVENT'] = 'Throw'
master_df.loc[master_df['EVENT'].str.contains(r'Javelin', na=True), 'CATEGORY_EVENT'] = 'Throw'
master_df.loc[master_df['EVENT'].str.contains(r'Shot', na=True), 'CATEGORY_EVENT'] = 'Throw'
master_df.loc[master_df['EVENT'].str.contains(r'Vault', na=True), 'CATEGORY_EVENT'] = 'Jump'
master_df.loc[master_df['EVENT'].str.contains(r'Throw', na=True), 'CATEGORY_EVENT'] = 'Throw'
master_df.loc[master_df['EVENT'].str.contains(r'Jump', na=True), 'CATEGORY_EVENT'] = 'Jump'
master_df.loc[master_df['EVENT'].str.contains(r'Relay', na=True), 'CATEGORY_EVENT'] = 'Relay'
master_df.loc[master_df['EVENT'].str.contains(r'Steeple', na=True), 'CATEGORY_EVENT'] = 'Steeple'
master_df.loc[master_df['EVENT'].str.contains(r'Walk', na=True), 'CATEGORY_EVENT'] = 'Walk'
master_df.loc[master_df['EVENT'].str.contains(r'Hurdles', na=True), 'CATEGORY_EVENT'] = 'Hurdles'
master_df.loc[master_df['EVENT'].str.contains(r'Pentathlon', na=True), 'CATEGORY_EVENT'] = 'Pentathlon'
master_df.loc[master_df['EVENT'].str.contains(r'Heptathlon', na=True), 'CATEGORY_EVENT'] = 'Heptathlon'
master_df.loc[master_df['EVENT'].str.contains(r'Triathlon', na=True), 'CATEGORY_EVENT'] = 'Triathlon'
master_df.loc[master_df['EVENT'].str.contains(r'Decathlon', na=True), 'CATEGORY_EVENT'] = 'Decathlon'
master_df.loc[master_df['EVENT'].str.contains(r'Marathon', na=True), 'CATEGORY_EVENT'] = 'Marathon'
master_df.loc[master_df['EVENT'].str.contains(r'4 x 100m', na=True), 'CATEGORY_EVENT'] = 'Relay'
master_df.loc[master_df['EVENT'].str.contains(r'4 x 400m', na=True), 'CATEGORY_EVENT'] = 'Relay'

# Road race

master_df.loc[master_df['EVENT'].str.contains(r'Road|road', na=True), 'CATEGORY_EVENT'] = 'Road'


# Copy relay event dictionary names into NAME column

master_df['NAME'] = np.where(((master_df['NAME'] == '') & ((master_df['EVENT']=='4 x 100m')|(master_df['EVENT']=='4 x 400m'))), master_df['DICT_RESULTS'], master_df['NAME'])

In [117]:
# Convert to NEW SCHEMA



master_df['LAST_NAME'] = ''
master_df['FIRST_NAME'] = ''
master_df['OTHER_NAME'] = ''
master_df['TEAM'] = ''
master_df['SEED'] = ''
master_df['LANE'] = ''
master_df['DIVISION'] = ''
master_df['AGE'] = ''
master_df['UNIQUE_ID'] = ''
master_df['ATHLETE_ID'] = ''
master_df['TIMESTAMP'] = ''
master_df['TAG_ID'] = ''
master_df['POINTS'] = ''
master_df['GROUP'] = ''
master_df['SESSION']=''
master_df['DISTANCE']=''


master_df = master_df.reindex(columns= ['FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
                                        'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'NATIONALITY',
                                        'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID',
                                        'SOURCE', 'REMARKS', 'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE', 'HOST_CITY', 'RX_TIME'])


In [118]:
SGP_athletes=master_df[master_df['NATIONALITY']=='SGP']

SGP_athletes.reset_index(drop=True, inplace=True)


In [119]:
SGP_athletes['EVENT_CLASS']

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
20   NaN
21   NaN
22   NaN
23   NaN
24   NaN
25   NaN
26   NaN
27   NaN
28   NaN
29   NaN
30   NaN
31   NaN
Name: EVENT_CLASS, dtype: float64

In [120]:
SGP_athletes['RESULT']=SGP_athletes['RESULT'].astype(str)

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_15156/4034045352.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  SGP_athletes['RESULT']=SGP_athletes['RESULT'].astype(str)


In [121]:
SGP_athletes['RESULT']

0       13.44
1       11.90
2       11.82
3       12.18
4       12.02
5       37.26
6       15.40
7       15.68
8       13.77
9        1.50
10    1:12:37
11      13.80
12    1:08:54
13      13.70
14      14.46
15    1:25:12
16    1:09:21
17    1:15:32
18    4:38.01
19      27.56
20      10.86
21      14.20
22      10.92
23      14.25
24      37:56
25      15.19
26       6.08
27      12.68
28      11.69
29      22.99
30    1:39:37
31      14.21
Name: RESULT, dtype: object

In [122]:
#os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2025/Aug/batch 2/')

#SGP_athletes.to_csv("check.csv", index=False, float_format="%.2f")


In [123]:
# Save to Excel

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug/batch2/')

SGP_athletes.to_excel("SGP_30Aug_extract_batch2.xlsx", index=False)

In [124]:
df = SGP_athletes

# Read in manually checked csv and upload into BQ

In [125]:
# Run this step if manually loading csv that has been modified after above step otherwise move onto next line

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug/batch2/')

athletes = pd.read_excel("SGP_30Aug_extract_batch2.xlsx")

df=athletes

In [126]:
df

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,SOURCE,REMARKS,TIMESTAMP,VENUE,SUB_EVENT,SESSION,EVENT_CLASS,DISTANCE,HOST_CITY,RX_TIME
0,NaN,NaN,NaN,Yao Peng Lim,7,NaN,NaN,NaN,13.44,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,SB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
1,NaN,NaN,NaN,Weiyi Lu,5,NaN,NaN,NaN,11.90,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
2,NaN,NaN,NaN,Shawn Wee,5,NaN,NaN,NaN,11.82,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
3,NaN,NaN,NaN,Danny Lum,4,NaN,NaN,NaN,12.18,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
4,NaN,NaN,NaN,Curtis Liau,4,NaN,NaN,NaN,12.02,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
5,NaN,NaN,NaN,Wei De Toh,6,NaN,NaN,NaN,37.26,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
6,NaN,NaN,NaN,Furene Wang,6,NaN,NaN,NaN,15.40,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
7,NaN,NaN,NaN,Anna Liisa Milani,6,NaN,NaN,NaN,15.68,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
8,NaN,NaN,NaN,Aaron Augustine Bing Kai Huang,6,NaN,NaN,NaN,13.77,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN
9,NaN,NaN,NaN,Zilin Jiang,3,NaN,NaN,NaN,1.50,NaN,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,NaN,events:,Multievents,NaN,NaN,NaN,Daegu,NaN


# If not coming in from a manually loaded csv

In [127]:
# SKIP if DATE column is already in datetime format
# Process DATE field into DD-MM-YY format

import datetime

for i in range(len(df)):
        
    rowIndex = df.index[i]

    date = str(df.loc[rowIndex,'DATE'])
    year = str(df.loc[rowIndex,'YEAR'])
    
 #   print(date)
 #   date_conv = pd.to_datetime(df.loc[rowIndex, 'DATE'], format='mixed')
 #   print(date_conv)
        
    if 'to' in date or ' - ' in date:
        
        if re.search('to|\s\-\s\d\s|\s\-\d\d', date):  # e.g. 03-04
              
            pos = re.search('to|\s\-\s\d', date)
            # Splice string to day and month

            split_pos_start=pos.start()+3

            final_date = date[split_pos_start:] # left string post splicing

            print(i, pos, date, final_date)
            final_year = year[2:]

            event_date = final_date + '/' + final_year

            print('event_date', event_date)

            df.loc[rowIndex, 'event_date'] = event_date

        elif re.search('(\-\s\d\w)|(\-\s\d\d\w)', date):  # e.g. 18 - 19 January
                        
            pos = re.search('\-', date)  # from '-' onwards only
            # Splice string to day and month

            split_pos_start=pos.start()+2
            

            final_date = date[split_pos_start:] # left string post splicing

            
            final_year = year[2:]

            event_date = final_date + ' ' + final_year


            df.loc[rowIndex, 'event_date'] = event_date
            
        
    elif re.search('\w\-\w', date):
        
        if df.loc[rowIndex, 'COMPETITION'] == "National School Games":
            
            if df.loc[rowIndex, 'YEAR'] == '2024':
        
                event_date = '04'+'/'+date[1:3] + '/' + year[2:]  # reverse order from dd/mm to mm/dd. 04 because event was in April 24 only
            
       #         print('NSG 2024', event_date)
        
                df.loc[rowIndex, 'event_date'] = event_date
            
            elif df.loc[rowIndex, 'YEAR'] == '2025':
                
                event_date = date + '-' +year[2:]
                
        #        print('NSG2025', event_date)
                
                df.loc[rowIndex, 'event_date'] = event_date
                
        elif re.search('\d\-\d',  date):        #10-13 April
            
         #   print('HERE', i, date)

            rpos = re.search('\-', date)
            string = date[rpos.end():]
            
            print('extracted date', string)
            
            event_date = string + ' ' + year
            
            print('event date', event_date)
            
            df.loc[rowIndex, 'event_date'] = event_date

        else:  # DD JAN-DEC YY format
            
            event_date = date + '-' + year[2:]
            
            df.loc[rowIndex, 'event_date'] = event_date
    
            
            
        
    else:  # DD JAN-DEC YY format
            
        event_date = date + '-' + year[2:]
            
        df.loc[rowIndex, 'event_date'] = event_date
        
for col in df.columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()

extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-24
event date 08-24 2026
extracted date 08-24
event date 08-24 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-23
event date 08-23 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-24
event date 08-24 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-23
event date 08-23 2026
extracted date 08-23
event date 08-23 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-22
event date 08-22 2026
extracted date 08-24
event date 08-24 2026
extracted date 08-23
event date 08-23 2026
extracted date 08-11
event date 08-11 2026
extracted date 08-11
event date 08-11 2026
extracted date 08-11
event date 08-11 2026
extracted d

In [128]:
# Replace nan with blanks

df = df.fillna('')


In [129]:
df['event_date_dt'] = pd.to_datetime(df['event_date'], format='mixed', dayfirst=False, utc=True)

# Copy over datetime with UTC back into DATE column

df['DATE'] = df['event_date_dt']

# Check and drop event_date column before saving

df.drop(['event_date', 'event_date_dt'], axis=1, inplace=True)

In [130]:
df['DATE'] = pd.to_datetime(df['DATE'], format='mixed', dayfirst=False, utc=True)


In [131]:
df['DATE']

0    2026-08-22 00:00:00+00:00
1    2026-08-22 00:00:00+00:00
2    2026-08-22 00:00:00+00:00
3    2026-08-22 00:00:00+00:00
4    2026-08-22 00:00:00+00:00
5    2026-08-22 00:00:00+00:00
6    2026-08-24 00:00:00+00:00
7    2026-08-24 00:00:00+00:00
8    2026-08-22 00:00:00+00:00
9    2026-08-23 00:00:00+00:00
10   2026-08-22 00:00:00+00:00
11   2026-08-24 00:00:00+00:00
12   2026-08-22 00:00:00+00:00
13   2026-08-23 00:00:00+00:00
14   2026-08-23 00:00:00+00:00
15   2026-08-22 00:00:00+00:00
16   2026-08-22 00:00:00+00:00
17   2026-08-22 00:00:00+00:00
18   2026-08-24 00:00:00+00:00
19   2026-08-23 00:00:00+00:00
20   2026-08-11 00:00:00+00:00
21   2026-08-11 00:00:00+00:00
22   2026-08-11 00:00:00+00:00
23   2026-08-11 00:00:00+00:00
24   2026-08-16 00:00:00+00:00
25   2026-08-24 00:00:00+00:00
26   2026-08-23 00:00:00+00:00
27   2026-08-24 00:00:00+00:00
28   2026-08-26 00:00:00+00:00
29   2026-08-26 00:00:00+00:00
30   2026-08-15 00:00:00+00:00
31   2026-08-08 00:00:00+00:00
Name: DA

In [132]:
df.loc[df['EVENT'].str.contains(r'10000m', na=True), 'EVENT'] = '10,000m'
df.loc[df['STAGE'].str.contains(r'Semi', na=True), 'STAGE'] = 'Semifinal'
df.loc[df['STAGE'].str.contains(r'Final', na=True), 'STAGE'] = 'Finals'


In [133]:
df['DATE']

0    2026-08-22 00:00:00+00:00
1    2026-08-22 00:00:00+00:00
2    2026-08-22 00:00:00+00:00
3    2026-08-22 00:00:00+00:00
4    2026-08-22 00:00:00+00:00
5    2026-08-22 00:00:00+00:00
6    2026-08-24 00:00:00+00:00
7    2026-08-24 00:00:00+00:00
8    2026-08-22 00:00:00+00:00
9    2026-08-23 00:00:00+00:00
10   2026-08-22 00:00:00+00:00
11   2026-08-24 00:00:00+00:00
12   2026-08-22 00:00:00+00:00
13   2026-08-23 00:00:00+00:00
14   2026-08-23 00:00:00+00:00
15   2026-08-22 00:00:00+00:00
16   2026-08-22 00:00:00+00:00
17   2026-08-22 00:00:00+00:00
18   2026-08-24 00:00:00+00:00
19   2026-08-23 00:00:00+00:00
20   2026-08-11 00:00:00+00:00
21   2026-08-11 00:00:00+00:00
22   2026-08-11 00:00:00+00:00
23   2026-08-11 00:00:00+00:00
24   2026-08-16 00:00:00+00:00
25   2026-08-24 00:00:00+00:00
26   2026-08-23 00:00:00+00:00
27   2026-08-24 00:00:00+00:00
28   2026-08-26 00:00:00+00:00
29   2026-08-26 00:00:00+00:00
30   2026-08-15 00:00:00+00:00
31   2026-08-08 00:00:00+00:00
Name: DA

In [134]:
df['TAG_ID']

0     nan
1     nan
2     nan
3     nan
4     nan
5     nan
6     nan
7     nan
8     nan
9     nan
10    nan
11    nan
12    nan
13    nan
14    nan
15    nan
16    nan
17    nan
18    nan
19    nan
20    nan
21    nan
22    nan
23    nan
24    nan
25    nan
26    nan
27    nan
28    nan
29    nan
30    nan
31    nan
Name: TAG_ID, dtype: object

In [135]:
# Rerun map general event category in case there are manual entries in file

df.loc[df['EVENT'].str.contains(r'^100m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'^400m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'^600m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'^60m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'^200m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'50m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'80m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'300m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Sprint'
df.loc[df['EVENT'].str.contains(r'^3000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
df.loc[df['EVENT'].str.contains(r'Mile', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'^5000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
df.loc[df['EVENT'].str.contains(r'^2000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'^1000m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'^800m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'^1500m$', regex=True, na=True), 'CATEGORY_EVENT'] = 'Mid'
df.loc[df['EVENT'].str.contains(r'10,000m', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
df.loc[df['EVENT'].str.contains(r'5km', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
df.loc[df['EVENT'].str.contains(r'10km', regex=True, na=True), 'CATEGORY_EVENT'] = 'Long'
df.loc[df['EVENT'].str.contains(r'Cross Country', na=True), 'CATEGORY_EVENT'] = 'Cross Country'


# Override values

df.loc[df['EVENT'].str.contains(r'Discus', na=True), 'CATEGORY_EVENT'] = 'Throw'
df.loc[df['EVENT'].str.contains(r'Javelin', na=True), 'CATEGORY_EVENT'] = 'Throw'
df.loc[df['EVENT'].str.contains(r'Shot', na=True), 'CATEGORY_EVENT'] = 'Throw'
df.loc[df['EVENT'].str.contains(r'Vault', na=True), 'CATEGORY_EVENT'] = 'Jump'
df.loc[df['EVENT'].str.contains(r'Throw', na=True), 'CATEGORY_EVENT'] = 'Throw'
df.loc[df['EVENT'].str.contains(r'Jump', na=True), 'CATEGORY_EVENT'] = 'Jump'
df.loc[df['EVENT'].str.contains(r'Relay', na=True), 'CATEGORY_EVENT'] = 'Relay'
df.loc[df['EVENT'].str.contains(r'Steeple', na=True), 'CATEGORY_EVENT'] = 'Steeple'
df.loc[df['EVENT'].str.contains(r'Walk', na=True), 'CATEGORY_EVENT'] = 'Walk'
df.loc[df['EVENT'].str.contains(r'Hurdles', na=True), 'CATEGORY_EVENT'] = 'Hurdles'
df.loc[df['EVENT'].str.contains(r'Pentathlon', na=True), 'CATEGORY_EVENT'] = 'Pentathlon'
df.loc[df['EVENT'].str.contains(r'Heptathlon', na=True), 'CATEGORY_EVENT'] = 'Heptathlon'
df.loc[df['EVENT'].str.contains(r'Triathlon', na=True), 'CATEGORY_EVENT'] = 'Triathlon'
df.loc[df['EVENT'].str.contains(r'Decathlon', na=True), 'CATEGORY_EVENT'] = 'Decathlon'
df.loc[df['EVENT'].str.contains(r'Marathon', na=True), 'CATEGORY_EVENT'] = 'Marathon'
df.loc[df['EVENT'].str.contains(r'4 x 100m', na=True), 'CATEGORY_EVENT'] = 'Relay'
df.loc[df['EVENT'].str.contains(r'4 x 400m', na=True), 'CATEGORY_EVENT'] = 'Relay'

# Road race

df.loc[df['EVENT'].str.contains(r'Road|road', na=True), 'CATEGORY_EVENT'] = 'Road'


# Copy relay event dictionary names into NAME column

#df['NAME'] = np.where(((df['NAME'] == '') & ((df['EVENT']=='4 x 100m')|(df['EVENT']=='4 x 400m'))), df['DICT_RESULTS'], df['NAME'])

In [136]:
df['DATE']

0    2026-08-22 00:00:00+00:00
1    2026-08-22 00:00:00+00:00
2    2026-08-22 00:00:00+00:00
3    2026-08-22 00:00:00+00:00
4    2026-08-22 00:00:00+00:00
5    2026-08-22 00:00:00+00:00
6    2026-08-24 00:00:00+00:00
7    2026-08-24 00:00:00+00:00
8    2026-08-22 00:00:00+00:00
9    2026-08-23 00:00:00+00:00
10   2026-08-22 00:00:00+00:00
11   2026-08-24 00:00:00+00:00
12   2026-08-22 00:00:00+00:00
13   2026-08-23 00:00:00+00:00
14   2026-08-23 00:00:00+00:00
15   2026-08-22 00:00:00+00:00
16   2026-08-22 00:00:00+00:00
17   2026-08-22 00:00:00+00:00
18   2026-08-24 00:00:00+00:00
19   2026-08-23 00:00:00+00:00
20   2026-08-11 00:00:00+00:00
21   2026-08-11 00:00:00+00:00
22   2026-08-11 00:00:00+00:00
23   2026-08-11 00:00:00+00:00
24   2026-08-16 00:00:00+00:00
25   2026-08-24 00:00:00+00:00
26   2026-08-23 00:00:00+00:00
27   2026-08-24 00:00:00+00:00
28   2026-08-26 00:00:00+00:00
29   2026-08-26 00:00:00+00:00
30   2026-08-15 00:00:00+00:00
31   2026-08-08 00:00:00+00:00
Name: DA

In [137]:
# ============================================================
# ATHLETE IDENTITY ENRICHMENT
# Standardise NAME and backfill UNIQUE_ID and DOB
# ============================================================
#
# Incoming DOB convention:
#     Primarily DD/MM/YYYY
#
# Internal DOB convention:
#     YYYY-MM-DD
#
# Final output DOB convention:
#     DD/MM/YYYY
#
# Matching priority:
#     1. Existing UNIQUE_ID
#     2. Exact name key + exact DOB
#     3. Unambiguous exact name key
#     4. Unambiguous token-order signature fallback
#
# The token-order fallback is deliberately conservative:
# it is used only when the same set of name tokens maps to
# exactly one UNIQUE_ID in the reference data.
#
# Authoritative fields:
#     athlete_master supplies:
#       - canonical NAME
#       - UNIQUE_ID
#       - DOB
#
# Important correction:
# If a name maps unambiguously to one UNIQUE_ID but the incoming
# DOB differs from the master DOB, the match is retained and the
# master DOB replaces the incoming DOB.
# ============================================================


# ------------------------------------------------------------
# 0. Imports
# ------------------------------------------------------------

import os
import re
import unicodedata

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

ATHLETE_MASTER_PATH = (
    "/Users/veesheenyuen/Desktop/DataScience/SAA/Unique_ID/"
    "athlete_master_latest_spliced.csv"
)

ATHLETE_VARIATIONS_PATH = (
    "/Users/veesheenyuen/Desktop/DataScience/SAA/Unique_ID/"
    "athlete_name_variations_latest_spliced.csv"
)

IDENTITY_AUDIT_OUTPUT = (
    "/Users/veesheenyuen/Desktop/DataScience/SAA/External events/2026/Aug/General/"
    "audit/identity_enrichment_audit.csv"
)

FINAL_OUTPUT_PATH = (
    "/Users/veesheenyuen/Desktop/DataScience/SAA/External events/2026/Aug/General/"
    "2025-26_competitions/final.csv"
)


# Final output format:
OUTPUT_DOB_FORMAT = "%d/%m/%Y"

# For YYYY/MM/DD instead, use:
# OUTPUT_DOB_FORMAT = "%Y/%m/%d"

# For BigQuery DATE-compatible output, use:
# OUTPUT_DOB_FORMAT = "%Y-%m-%d"


# ------------------------------------------------------------
# 2. General cleaning helpers
# ------------------------------------------------------------

NULL_LIKE_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "nat",
    "<na>",
}


def clean_base_name(value):
    """
    Clean an individual string value.
    """
    if pd.isna(value):
        return ""

    value = str(value)
    value = value.replace("\xa0", " ")

    # Remove control characters.
    value = re.sub(
        r"[\x00-\x1f\x7f-\x9f]",
        "",
        value,
    )

    value = value.replace("\r", " ")
    value = value.replace("\n", " ")

    # Collapse repeated whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value,
    )

    return value.strip()


def clean_string_col(series):
    """
    Clean an entire pandas string column.
    """
    series = (
        series
        .fillna("")
        .astype(str)
        .str.replace(
            "\xa0",
            " ",
            regex=False,
        )
        .str.replace(
            r"[\x00-\x1f\x7f-\x9f]",
            "",
            regex=True,
        )
        .str.replace(
            "\r",
            " ",
            regex=False,
        )
        .str.replace(
            "\n",
            " ",
            regex=False,
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )

    null_mask = (
        series
        .str.casefold()
        .isin(NULL_LIKE_VALUES)
    )

    series.loc[null_mask] = ""

    return series


def name_match_key(value):
    """
    Generate a punctuation-insensitive and whitespace-insensitive
    athlete-name lookup key.

    Examples:
        Louis, Marc Brian
            -> louismarcbrian

        Muhammad Rashid, Emir
            -> muhammadrashidemir

        ^Chen Xiang Ang$
            -> chenxiangang
    """
    value = clean_base_name(value)

    # Remove regex anchors and escape characters.
    value = value.replace("^", "")
    value = value.replace("$", "")
    value = value.replace("\\", "")

    value = unicodedata.normalize(
        "NFKD",
        value,
    )

    # Retain letters and numbers only.
    value = "".join(
        character
        for character in value
        if character.isalnum()
    )

    return value.casefold()


def name_token_signature(value):
    """
    Generate an order-insensitive token signature.

    Examples:
        'Shyen Joshua Lee'
        'Joshua Shyen Lee'
        'Lee Joshua Shyen'

    all become:
        'joshua|lee|shyen'

    This is used only as a fallback after exact name-key
    matching. Ambiguous signatures are never auto-resolved.
    """
    value = clean_base_name(value)

    value = value.replace("^", "")
    value = value.replace("$", "")
    value = value.replace("\\", "")

    value = unicodedata.normalize(
        "NFKD",
        value,
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        value.casefold(),
    )

    return "|".join(
        sorted(tokens)
    )


def require_columns(
    dataframe,
    required_columns,
    dataframe_name,
):
    """
    Confirm that required columns exist.
    """
    missing_columns = (
        set(required_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )


def ordered_nonblank_unique(series):
    """
    Return unique, nonblank strings while preserving order.
    """
    output = []
    seen = set()

    for value in series:
        value = clean_base_name(value)

        if not value or value in seen:
            continue

        seen.add(value)
        output.append(value)

    return output


# ------------------------------------------------------------
# 3. DOB helpers
# ------------------------------------------------------------

def clean_dob_text(value):
    """
    Convert a DOB value into a clean string.
    """
    value = clean_base_name(value)

    if value.casefold() in NULL_LIKE_VALUES:
        return ""

    # Do not treat a year-only value as a complete DOB.
    if re.fullmatch(r"\d{4}", value):
        return ""

    return value


def parse_date_with_format(
    value,
    date_format,
):
    """
    Parse one date using an explicit format.

    Return:
        YYYY-MM-DD, or blank if unparseable.
    """
    parsed = pd.to_datetime(
        value,
        format=date_format,
        errors="coerce",
    )

    if pd.isna(parsed):
        return ""

    return parsed.strftime("%Y-%m-%d")


def normalize_reference_dob(value):
    """
    Normalise DOB values from athlete_master and
    athlete_name_variations.

    Reference DOBs normally use YYYY-MM-DD.

    Return:
        YYYY-MM-DD
    """
    value = clean_dob_text(value)

    if not value:
        return ""

    # Handle pandas timestamps converted to strings.
    timestamp_match = re.fullmatch(
        r"(\d{4}-\d{1,2}-\d{1,2})(?:\s+.*)?",
        value,
    )

    if timestamp_match:
        return parse_date_with_format(
            timestamp_match.group(1),
            "%Y-%m-%d",
        )

    # YYYY/MM/DD
    if re.fullmatch(
        r"\d{4}/\d{1,2}/\d{1,2}",
        value,
    ):
        return parse_date_with_format(
            value,
            "%Y/%m/%d",
        )

    # YYYYMMDD
    if re.fullmatch(r"\d{8}", value):
        first_four = int(value[:4])

        if 1900 <= first_four <= 2100:
            return parse_date_with_format(
                value,
                "%Y%m%d",
            )

    # DD/MM/YYYY fallback.
    if re.fullmatch(
        r"\d{1,2}/\d{1,2}/\d{4}",
        value,
    ):
        return parse_date_with_format(
            value,
            "%d/%m/%Y",
        )

    # DD-MM-YYYY fallback.
    if re.fullmatch(
        r"\d{1,2}-\d{1,2}-\d{4}",
        value,
    ):
        return parse_date_with_format(
            value,
            "%d-%m-%Y",
        )

    parsed = pd.to_datetime(
        value,
        errors="coerce",
        dayfirst=True,
    )

    if pd.isna(parsed):
        return ""

    return parsed.strftime("%Y-%m-%d")


def normalize_incoming_dob(value):
    """
    Normalise incoming competition DOBs.

    Primary source convention:
        DD/MM/YYYY

    For impossible DD/MM/YYYY values such as 01/22/2009,
    the function safely interprets the value as MM/DD/YYYY.

    Ambiguous values where both components are 12 or below
    follow the stated source convention: DD/MM/YYYY.

    Return:
        YYYY-MM-DD
    """
    value = clean_dob_text(value)

    if not value:
        return ""

    # Handle pandas timestamps or ISO strings.
    timestamp_match = re.fullmatch(
        r"(\d{4}-\d{1,2}-\d{1,2})(?:\s+.*)?",
        value,
    )

    if timestamp_match:
        return parse_date_with_format(
            timestamp_match.group(1),
            "%Y-%m-%d",
        )

    # YYYY/MM/DD
    if re.fullmatch(
        r"\d{4}/\d{1,2}/\d{1,2}",
        value,
    ):
        return parse_date_with_format(
            value,
            "%Y/%m/%d",
        )

    # Slash-separated date.
    slash_match = re.fullmatch(
        r"(\d{1,2})/(\d{1,2})/(\d{4})",
        value,
    )

    if slash_match:
        first = int(slash_match.group(1))
        second = int(slash_match.group(2))

        # Clearly DD/MM/YYYY:
        # first component is greater than 12.
        if first > 12 and second <= 12:
            return parse_date_with_format(
                value,
                "%d/%m/%Y",
            )

        # Clearly MM/DD/YYYY:
        # second component is greater than 12.
        if second > 12 and first <= 12:
            return parse_date_with_format(
                value,
                "%m/%d/%Y",
            )

        # Ambiguous: use the declared DD/MM/YYYY convention.
        return parse_date_with_format(
            value,
            "%d/%m/%Y",
        )

    # Dash-separated date.
    dash_match = re.fullmatch(
        r"(\d{1,2})-(\d{1,2})-(\d{4})",
        value,
    )

    if dash_match:
        first = int(dash_match.group(1))
        second = int(dash_match.group(2))

        if first > 12 and second <= 12:
            return parse_date_with_format(
                value,
                "%d-%m-%Y",
            )

        if second > 12 and first <= 12:
            return parse_date_with_format(
                value,
                "%m-%d-%Y",
            )

        return parse_date_with_format(
            value,
            "%d-%m-%Y",
        )

    # Compact YYYYMMDD.
    if re.fullmatch(r"\d{8}", value):
        first_four = int(value[:4])

        if 1900 <= first_four <= 2100:
            return parse_date_with_format(
                value,
                "%Y%m%d",
            )

    # Final fallback follows the declared DD/MM convention.
    parsed = pd.to_datetime(
        value,
        errors="coerce",
        dayfirst=True,
    )

    if pd.isna(parsed):
        return ""

    return parsed.strftime("%Y-%m-%d")


def format_output_dob(value):
    """
    Format a DOB using OUTPUT_DOB_FORMAT.

    Internally, DOB values should normally be YYYY-MM-DD.
    """
    value = clean_dob_text(value)

    if not value:
        return ""

    normalized = normalize_reference_dob(value)

    if not normalized:
        normalized = normalize_incoming_dob(value)

    if not normalized:
        return value

    parsed = pd.to_datetime(
        normalized,
        format="%Y-%m-%d",
        errors="coerce",
    )

    if pd.isna(parsed):
        return value

    return parsed.strftime(
        OUTPUT_DOB_FORMAT
    )


# ------------------------------------------------------------
# 4. Load identity reference files
# ------------------------------------------------------------

athlete_master = pd.read_csv(
    ATHLETE_MASTER_PATH,
    dtype=str,
).fillna("")

athlete_variations = pd.read_csv(
    ATHLETE_VARIATIONS_PATH,
    dtype=str,
).fillna("")


# Clean reference headers.
athlete_master.columns = (
    athlete_master
    .columns
    .astype(str)
    .str.strip()
)

athlete_variations.columns = (
    athlete_variations
    .columns
    .astype(str)
    .str.strip()
)


required_master_columns = {
    "UNIQUE_ID",
    "CANONICAL_NAME",
}

required_variation_columns = {
    "VARIATION_NAME",
    "VARIATION_KEY",
    "UNIQUE_ID",
    "CANONICAL_NAME",
}


require_columns(
    athlete_master,
    required_master_columns,
    "athlete_master",
)

require_columns(
    athlete_variations,
    required_variation_columns,
    "athlete_name_variations",
)


# ------------------------------------------------------------
# 5. Keep active reference records only
# ------------------------------------------------------------

active_values = {
    "TRUE",
    "YES",
    "1",
    "",
}


if "ACTIVE" in athlete_master.columns:
    athlete_master = athlete_master[
        athlete_master["ACTIVE"]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(active_values)
    ].copy()


if "ACTIVE" in athlete_variations.columns:
    athlete_variations = athlete_variations[
        athlete_variations["ACTIVE"]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(active_values)
    ].copy()


# ------------------------------------------------------------
# 6. Normalise athlete_master
# ------------------------------------------------------------

athlete_master["UNIQUE_ID"] = clean_string_col(
    athlete_master["UNIQUE_ID"]
)

athlete_master["CANONICAL_NAME"] = clean_string_col(
    athlete_master["CANONICAL_NAME"]
)


if "DOB" not in athlete_master.columns:
    athlete_master["DOB"] = ""


athlete_master["DOB_NORMALIZED"] = (
    athlete_master["DOB"]
    .apply(normalize_reference_dob)
)

athlete_master["MASTER_NAME_KEY"] = (
    athlete_master["CANONICAL_NAME"]
    .apply(name_match_key)
)

athlete_master["MASTER_TOKEN_SIGNATURE"] = (
    athlete_master["CANONICAL_NAME"]
    .apply(name_token_signature)
)


# Remove master rows without a usable UID.
athlete_master = athlete_master[
    athlete_master["UNIQUE_ID"] != ""
].copy()


# ------------------------------------------------------------
# 6A. Validate athlete_master UNIQUE_ID consistency
# ------------------------------------------------------------
#
# One UNIQUE_ID must not point to multiple canonical names or
# multiple nonblank DOBs. Do this BEFORE any drop_duplicates()
# operation so reference-data conflicts cannot be hidden.

master_uid_conflicts = (
    athlete_master
    .groupby(
        "UNIQUE_ID",
        dropna=False,
    )
    .agg(
        CANONICAL_NAME_COUNT=(
            "CANONICAL_NAME",
            lambda s: len({
                clean_base_name(v).casefold()
                for v in s
                if clean_base_name(v)
            }),
        ),
        DOB_COUNT=(
            "DOB_NORMALIZED",
            lambda s: len({
                clean_base_name(v)
                for v in s
                if clean_base_name(v)
            }),
        ),
    )
    .reset_index()
)

master_uid_conflicts = master_uid_conflicts.loc[
    (master_uid_conflicts["CANONICAL_NAME_COUNT"] > 1)
    | (master_uid_conflicts["DOB_COUNT"] > 1)
].copy()

if not master_uid_conflicts.empty:
    conflicting_uids = set(
        master_uid_conflicts["UNIQUE_ID"]
    )

    master_uid_conflict_details = (
        athlete_master.loc[
            athlete_master["UNIQUE_ID"].isin(
                conflicting_uids
            ),
            [
                "UNIQUE_ID",
                "CANONICAL_NAME",
                "DOB",
                "DOB_NORMALIZED",
            ],
        ]
        .sort_values(
            [
                "UNIQUE_ID",
                "CANONICAL_NAME",
                "DOB_NORMALIZED",
            ]
        )
    )

    display(master_uid_conflict_details)

    raise ValueError(
        "athlete_master contains conflicting records for the "
        "same UNIQUE_ID. Resolve these conflicts before upload."
    )


# ------------------------------------------------------------
# 7. Normalise athlete_name_variations
# ------------------------------------------------------------

athlete_variations["UNIQUE_ID"] = clean_string_col(
    athlete_variations["UNIQUE_ID"]
)

athlete_variations["VARIATION_NAME"] = clean_string_col(
    athlete_variations["VARIATION_NAME"]
)

athlete_variations["CANONICAL_NAME"] = clean_string_col(
    athlete_variations["CANONICAL_NAME"]
)


if "DOB" not in athlete_variations.columns:
    athlete_variations["DOB"] = ""


athlete_variations["DOB_NORMALIZED"] = (
    athlete_variations["DOB"]
    .apply(normalize_reference_dob)
)


# Rebuild keys defensively.
athlete_variations["VARIATION_KEY"] = (
    athlete_variations["VARIATION_NAME"]
    .apply(name_match_key)
)


athlete_variations["TOKEN_SIGNATURE"] = (
    athlete_variations["VARIATION_NAME"]
    .apply(name_token_signature)
)


# ------------------------------------------------------------
# 8. Add canonical names as valid lookup names
# ------------------------------------------------------------

canonical_lookup = athlete_master[
    [
        "UNIQUE_ID",
        "CANONICAL_NAME",
        "DOB_NORMALIZED",
        "MASTER_NAME_KEY",
        "MASTER_TOKEN_SIGNATURE",
    ]
].copy()


canonical_lookup = canonical_lookup.rename(
    columns={
        "MASTER_NAME_KEY": "VARIATION_KEY",
        "MASTER_TOKEN_SIGNATURE": "TOKEN_SIGNATURE",
    }
)


canonical_lookup["VARIATION_NAME"] = (
    canonical_lookup["CANONICAL_NAME"]
)

canonical_lookup["MATCH_SOURCE"] = (
    "athlete_master_canonical_name"
)


variation_lookup = athlete_variations[
    [
        "VARIATION_NAME",
        "VARIATION_KEY",
        "TOKEN_SIGNATURE",
        "CANONICAL_NAME",
        "UNIQUE_ID",
        "DOB_NORMALIZED",
    ]
].copy()


variation_lookup["MATCH_SOURCE"] = (
    "athlete_name_variations"
)


identity_lookup = pd.concat(
    [
        variation_lookup,
        canonical_lookup[
            [
                "VARIATION_NAME",
                "VARIATION_KEY",
                "TOKEN_SIGNATURE",
                "CANONICAL_NAME",
                "UNIQUE_ID",
                "DOB_NORMALIZED",
                "MATCH_SOURCE",
            ]
        ],
    ],
    ignore_index=True,
)


identity_lookup = identity_lookup[
    (identity_lookup["VARIATION_KEY"] != "")
    & (identity_lookup["UNIQUE_ID"] != "")
].copy()


identity_lookup = identity_lookup.drop_duplicates()


# ------------------------------------------------------------
# 9. Attach authoritative athlete_master values
# ------------------------------------------------------------

master_uid_lookup = (
    athlete_master[
        [
            "UNIQUE_ID",
            "CANONICAL_NAME",
            "DOB_NORMALIZED",
        ]
    ]
    .drop_duplicates(
        subset=["UNIQUE_ID"],
        keep="first",
    )
)


identity_lookup = identity_lookup.merge(
    master_uid_lookup,
    on="UNIQUE_ID",
    how="left",
    suffixes=(
        "",
        "_MASTER",
    ),
)


identity_lookup["CANONICAL_NAME_FINAL"] = np.where(
    identity_lookup[
        "CANONICAL_NAME_MASTER"
    ].astype(str).str.strip() != "",
    identity_lookup[
        "CANONICAL_NAME_MASTER"
    ],
    identity_lookup[
        "CANONICAL_NAME"
    ],
)


identity_lookup["DOB_FINAL"] = np.where(
    identity_lookup[
        "DOB_NORMALIZED_MASTER"
    ].astype(str).str.strip() != "",
    identity_lookup[
        "DOB_NORMALIZED_MASTER"
    ],
    identity_lookup[
        "DOB_NORMALIZED"
    ],
)


# ------------------------------------------------------------
# 10. Detect ambiguous name keys
# ------------------------------------------------------------

key_uid_counts = (
    identity_lookup
    .groupby("VARIATION_KEY")["UNIQUE_ID"]
    .nunique()
    .reset_index(
        name="UID_CANDIDATE_COUNT"
    )
)


identity_lookup = identity_lookup.merge(
    key_uid_counts,
    on="VARIATION_KEY",
    how="left",
)


# Token-order signatures are a fallback only. A signature is
# safe for automatic use only when it maps to exactly one UID.
token_signature_uid_counts = (
    identity_lookup.loc[
        identity_lookup["TOKEN_SIGNATURE"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    ]
    .groupby("TOKEN_SIGNATURE")["UNIQUE_ID"]
    .nunique()
    .reset_index(
        name="TOKEN_UID_CANDIDATE_COUNT"
    )
)

identity_lookup = identity_lookup.merge(
    token_signature_uid_counts,
    on="TOKEN_SIGNATURE",
    how="left",
)


# ------------------------------------------------------------
# 11. Prepare incoming dataframe
# ------------------------------------------------------------

if "NAME" not in df.columns:
    raise ValueError(
        "Incoming df must contain a NAME column "
        "before identity enrichment."
    )


if "UNIQUE_ID" not in df.columns:
    df["UNIQUE_ID"] = ""


if "DOB" not in df.columns:
    df["DOB"] = ""


df["NAME"] = clean_string_col(
    df["NAME"]
)

df["UNIQUE_ID"] = clean_string_col(
    df["UNIQUE_ID"]
)

df["DOB"] = clean_string_col(
    df["DOB"]
)


# Preserve raw values for the audit.
df["_RAW_NAME"] = df["NAME"].copy()

df["_RAW_UNIQUE_ID"] = (
    df["UNIQUE_ID"].copy()
)

df["_RAW_DOB"] = df["DOB"].copy()


# Internal matching fields.
df["_NAME_MATCH_KEY"] = (
    df["NAME"]
    .apply(name_match_key)
)

df["_NAME_TOKEN_SIGNATURE"] = (
    df["NAME"]
    .apply(name_token_signature)
)

df["_DOB_NORMALIZED"] = (
    df["DOB"]
    .apply(normalize_incoming_dob)
)


# Initialise audit fields.
df["_MATCH_STATUS"] = "NO_MATCH"
df["_MATCH_SOURCE"] = ""
df["_MATCH_REASON"] = ""
df["_CANDIDATE_UNIQUE_IDS"] = ""
df["_CANDIDATE_DOBS"] = ""
df["_CANDIDATE_NAMES"] = ""


# ------------------------------------------------------------
# 12. Build lookup dictionaries
# ------------------------------------------------------------

master_by_uid = (
    athlete_master
    .drop_duplicates(
        subset=["UNIQUE_ID"],
        keep="first",
    )
    .set_index("UNIQUE_ID")
    .to_dict("index")
)


lookup_by_key = {
    key: group.copy()
    for key, group
    in identity_lookup.groupby(
        "VARIATION_KEY",
        sort=False,
    )
}


# Keep all token-signature groups for diagnostics, but automatic
# fallback is allowed only for signatures mapping to one UID.
lookup_by_token_signature_all = {
    signature: group.copy()
    for signature, group
    in identity_lookup.loc[
        identity_lookup["TOKEN_SIGNATURE"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    ].groupby(
        "TOKEN_SIGNATURE",
        sort=False,
    )
}

safe_token_signatures = set(
    token_signature_uid_counts.loc[
        token_signature_uid_counts[
            "TOKEN_UID_CANDIDATE_COUNT"
        ] == 1,
        "TOKEN_SIGNATURE",
    ]
)

ambiguous_token_signatures = set(
    token_signature_uid_counts.loc[
        token_signature_uid_counts[
            "TOKEN_UID_CANDIDATE_COUNT"
        ] > 1,
        "TOKEN_SIGNATURE",
    ]
)


# ------------------------------------------------------------
# 13. Row-level resolver
# ------------------------------------------------------------

def resolve_identity(row):
    """
    Resolve one incoming athlete row.
    """
    raw_uid = clean_base_name(
        row["_RAW_UNIQUE_ID"]
    )

    if raw_uid.casefold() in NULL_LIKE_VALUES:
        raw_uid = ""

    raw_dob = clean_base_name(
        row["_DOB_NORMALIZED"]
    )

    name_key = clean_base_name(
        row["_NAME_MATCH_KEY"]
    )

    token_signature = clean_base_name(
        row["_NAME_TOKEN_SIGNATURE"]
    )

    result = {
        "NAME": row["NAME"],
        "UNIQUE_ID": row["UNIQUE_ID"],
        "DOB": row["DOB"],
        "_MATCH_STATUS": "NO_MATCH",
        "_MATCH_SOURCE": "",
        "_MATCH_REASON": "",
        "_CANDIDATE_UNIQUE_IDS": "",
        "_CANDIDATE_DOBS": "",
        "_CANDIDATE_NAMES": "",
    }

    # --------------------------------------------------------
    # Case 1: Incoming UNIQUE_ID already exists
    # --------------------------------------------------------

    if raw_uid:
        if raw_uid not in master_by_uid:
            result["_MATCH_STATUS"] = (
                "UNKNOWN_UNIQUE_ID"
            )

            result["_MATCH_REASON"] = (
                "Incoming UNIQUE_ID not found "
                "in athlete_master"
            )

            return result

        master_record = master_by_uid[
            raw_uid
        ]

        master_name = clean_base_name(
            master_record.get(
                "CANONICAL_NAME",
                "",
            )
        )

        master_dob = clean_base_name(
            master_record.get(
                "DOB_NORMALIZED",
                "",
            )
        )

        result["UNIQUE_ID"] = raw_uid

        if master_name:
            result["NAME"] = master_name

        # athlete_master is authoritative.
        if master_dob:
            result["DOB"] = master_dob

        if (
            raw_dob
            and master_dob
            and raw_dob != master_dob
        ):
            result["_MATCH_STATUS"] = (
                "UID_DOB_CONFLICT_CORRECTED"
            )

            result["_MATCH_REASON"] = (
                f"Incoming DOB {raw_dob} differed "
                f"from master DOB {master_dob}; "
                f"final DOB replaced with the "
                f"athlete_master DOB."
            )
        else:
            result["_MATCH_STATUS"] = (
                "MATCHED_BY_UID"
            )

            result["_MATCH_REASON"] = (
                "Incoming UNIQUE_ID found in "
                "athlete_master"
            )

        result["_MATCH_SOURCE"] = (
            "athlete_master"
        )

        return result

    # --------------------------------------------------------
    # Case 2: UNIQUE_ID is missing
    # --------------------------------------------------------

    candidates = lookup_by_key.get(
        name_key
    )

    match_method = "EXACT_NAME_KEY"

    # Exact matching always takes precedence.
    # Only if exact matching fails do we consider the
    # order-insensitive token signature.
    if candidates is None or candidates.empty:

        if (
            token_signature
            and token_signature in safe_token_signatures
        ):
            candidates = (
                lookup_by_token_signature_all[
                    token_signature
                ].copy()
            )

            match_method = (
                "TOKEN_ORDER_FALLBACK"
            )

        elif (
            token_signature
            and token_signature in ambiguous_token_signatures
        ):
            ambiguous_candidates = (
                lookup_by_token_signature_all[
                    token_signature
                ].copy()
            )

            result["_MATCH_STATUS"] = (
                "MULTIPLE_UID_CANDIDATES"
            )

            result["_MATCH_REASON"] = (
                "No exact name-key match. Token-order signature "
                "maps to more than one UNIQUE_ID, so fallback "
                "was not applied."
            )

            result["_CANDIDATE_UNIQUE_IDS"] = "; ".join(
                ordered_nonblank_unique(
                    ambiguous_candidates["UNIQUE_ID"]
                )
            )

            result["_CANDIDATE_DOBS"] = "; ".join(
                ordered_nonblank_unique(
                    ambiguous_candidates["DOB_FINAL"]
                )
            )

            result["_CANDIDATE_NAMES"] = "; ".join(
                ordered_nonblank_unique(
                    ambiguous_candidates[
                        "CANONICAL_NAME_FINAL"
                    ]
                )
            )

            return result

        else:
            result["_MATCH_STATUS"] = "NO_MATCH"

            result["_MATCH_REASON"] = (
                "No exact name variation/canonical-name match "
                "and no safe token-order fallback found"
            )

            return result

    # Capture candidate information before DOB filtering.
    candidate_uids = ordered_nonblank_unique(
        candidates["UNIQUE_ID"]
    )

    candidate_dobs = ordered_nonblank_unique(
        candidates["DOB_FINAL"]
    )

    candidate_names = ordered_nonblank_unique(
        candidates["CANONICAL_NAME_FINAL"]
    )

    result["_CANDIDATE_UNIQUE_IDS"] = (
        "; ".join(candidate_uids)
    )

    result["_CANDIDATE_DOBS"] = (
        "; ".join(candidate_dobs)
    )

    result["_CANDIDATE_NAMES"] = (
        "; ".join(candidate_names)
    )

    # --------------------------------------------------------
    # Prefer an exact DOB candidate.
    #
    # If no DOB agrees, retain all name candidates.
    # This prevents a valid unambiguous name match from being
    # deleted solely because of a DOB discrepancy.
    # --------------------------------------------------------

    if raw_dob:
        exact_dob_candidates = candidates[
            candidates["DOB_FINAL"] == raw_dob
        ].copy()

        if not exact_dob_candidates.empty:
            candidates_to_use = (
                exact_dob_candidates
            )
        else:
            candidates_to_use = (
                candidates.copy()
            )
    else:
        candidates_to_use = (
            candidates.copy()
        )

    unique_ids = ordered_nonblank_unique(
        candidates_to_use["UNIQUE_ID"]
    )

    # --------------------------------------------------------
    # Exactly one UID candidate
    # --------------------------------------------------------

    if len(unique_ids) == 1:
        chosen_uid = unique_ids[0]

        chosen_rows = candidates_to_use[
            candidates_to_use[
                "UNIQUE_ID"
            ] == chosen_uid
        ].copy()

        chosen = chosen_rows.iloc[0]

        canonical_name = clean_base_name(
            chosen.get(
                "CANONICAL_NAME_FINAL",
                "",
            )
        )

        match_source = clean_base_name(
            chosen.get(
                "MATCH_SOURCE",
                "identity_lookup",
            )
        )

        reference_dobs = ordered_nonblank_unique(
            chosen_rows["DOB_FINAL"]
        )

        reference_dob = (
            reference_dobs[0]
            if reference_dobs
            else ""
        )

        result["UNIQUE_ID"] = chosen_uid

        if canonical_name:
            result["NAME"] = canonical_name

        # athlete_master is authoritative.
        # Replace the incoming DOB whenever a valid master DOB exists.
        if reference_dob:
            result["DOB"] = reference_dob

        if raw_dob:
            if (
                reference_dob
                and raw_dob == reference_dob
            ):
                result["_MATCH_STATUS"] = (
                    "MATCHED_BY_TOKEN_ORDER_DOB"
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else "MATCHED_BY_NAME_DOB"
                )

                result["_MATCH_REASON"] = (
                    "Token-order fallback matched one UID and "
                    "DOB confirmed"
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else "Exact name key matched and DOB confirmed"
                )

            elif not reference_dob:
                result["_MATCH_STATUS"] = (
                    "MATCHED_BY_TOKEN_ORDER_"
                    "DOB_UNAVAILABLE_IN_REFERENCE"
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else
                    "MATCHED_BY_NAME_ONLY_"
                    "DOB_UNAVAILABLE_IN_REFERENCE"
                )

                result["_MATCH_REASON"] = (
                    "Token-order fallback matched one UID; "
                    "incoming DOB present but reference DOB blank"
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else
                    "Exact name key matched one UID; "
                    "incoming DOB present but reference DOB blank"
                )

            else:
                result["_MATCH_STATUS"] = (
                    "TOKEN_ORDER_DOB_MISMATCH_CORRECTED"
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else "NAME_MATCH_DOB_MISMATCH_CORRECTED"
                )

                result["_MATCH_REASON"] = (
                    (
                        "Token-order fallback matched one UID. "
                        f"Incoming DOB {raw_dob} differed from "
                        f"reference DOB {reference_dob}; final DOB "
                        "replaced with the athlete_master DOB."
                    )
                    if match_method == "TOKEN_ORDER_FALLBACK"
                    else (
                        "Exact name key matched one UID. "
                        f"Incoming DOB {raw_dob} differed from "
                        f"reference DOB {reference_dob}; final DOB "
                        "replaced with the athlete_master DOB."
                    )
                )
        else:
            result["_MATCH_STATUS"] = (
                "MATCHED_BY_TOKEN_ORDER"
                if match_method == "TOKEN_ORDER_FALLBACK"
                else "MATCHED_BY_NAME_ONLY"
            )

            result["_MATCH_REASON"] = (
                "Token-order fallback matched exactly one "
                "UNIQUE_ID; incoming DOB blank"
                if match_method == "TOKEN_ORDER_FALLBACK"
                else
                "Exact name key matched exactly one "
                "UNIQUE_ID; incoming DOB blank"
            )

        result["_MATCH_SOURCE"] = (
            (
                f"token_order_fallback:{match_source}"
                if match_method == "TOKEN_ORDER_FALLBACK"
                else match_source
            )
        )

        return result

    # --------------------------------------------------------
    # More than one UID candidate
    # --------------------------------------------------------

    if len(unique_ids) > 1:
        result["_MATCH_STATUS"] = (
            "MULTIPLE_UID_CANDIDATES"
        )

        result["_MATCH_REASON"] = (
            "Name key maps to more than one "
            "possible UNIQUE_ID"
        )

        return result

    # --------------------------------------------------------
    # Name matched but no usable UID exists
    # --------------------------------------------------------

    result["_MATCH_STATUS"] = (
        "MATCHED_BUT_NO_UID"
    )

    result["_MATCH_REASON"] = (
        "Name key matched but no usable "
        "UNIQUE_ID candidate"
    )

    return result


# ------------------------------------------------------------
# 14. Apply resolver
# ------------------------------------------------------------

resolved = df.apply(
    resolve_identity,
    axis=1,
    result_type="expand",
)


# Preserve dataframe index alignment.
resolved.index = df.index


df["NAME"] = resolved["NAME"]
df["UNIQUE_ID"] = resolved["UNIQUE_ID"]
df["DOB"] = resolved["DOB"]

df["_MATCH_STATUS"] = (
    resolved["_MATCH_STATUS"]
)

df["_MATCH_SOURCE"] = (
    resolved["_MATCH_SOURCE"]
)

df["_MATCH_REASON"] = (
    resolved["_MATCH_REASON"]
)

df["_CANDIDATE_UNIQUE_IDS"] = (
    resolved["_CANDIDATE_UNIQUE_IDS"]
)

df["_CANDIDATE_DOBS"] = (
    resolved["_CANDIDATE_DOBS"]
)

df["_CANDIDATE_NAMES"] = (
    resolved["_CANDIDATE_NAMES"]
)


# ------------------------------------------------------------
# 15. Format final DOB values
# ------------------------------------------------------------

df["DOB"] = df["DOB"].apply(
    format_output_dob
)


# ------------------------------------------------------------
# 16. Build audit dataframe
# ------------------------------------------------------------

identity_audit = pd.DataFrame(
    {
        "ROW_INDEX": df.index,

        "RAW_NAME": (
            df["_RAW_NAME"]
        ),

        "STANDARDIZED_NAME": (
            df["NAME"]
        ),

        "RAW_UNIQUE_ID": (
            df["_RAW_UNIQUE_ID"]
        ),

        "FINAL_UNIQUE_ID": (
            df["UNIQUE_ID"]
        ),

        "RAW_DOB": (
            df["_RAW_DOB"]
        ),

        "NORMALIZED_INCOMING_DOB": (
            df["_DOB_NORMALIZED"]
        ),

        "FINAL_DOB": (
            df["DOB"]
        ),

        "NAME_MATCH_KEY": (
            df["_NAME_MATCH_KEY"]
        ),

        "NAME_TOKEN_SIGNATURE": (
            df["_NAME_TOKEN_SIGNATURE"]
        ),

        "MATCH_STATUS": (
            df["_MATCH_STATUS"]
        ),

        "MATCH_SOURCE": (
            df["_MATCH_SOURCE"]
        ),

        "MATCH_REASON": (
            df["_MATCH_REASON"]
        ),

        "CANDIDATE_UNIQUE_IDS": (
            df["_CANDIDATE_UNIQUE_IDS"]
        ),

        "CANDIDATE_DOBS": (
            df["_CANDIDATE_DOBS"]
        ),

        "CANDIDATE_NAMES": (
            df["_CANDIDATE_NAMES"]
        ),

        "COMPETITION": (
            df["COMPETITION"]
            if "COMPETITION" in df.columns
            else ""
        ),

        "EVENT": (
            df["EVENT"]
            if "EVENT" in df.columns
            else ""
        ),

        "DATE": (
            df["DATE"]
            if "DATE" in df.columns
            else ""
        ),

        "RESULT": (
            df["RESULT"]
            if "RESULT" in df.columns
            else ""
        ),
    }
)


# ------------------------------------------------------------
# 17. Save audit
# ------------------------------------------------------------

audit_directory = os.path.dirname(
    IDENTITY_AUDIT_OUTPUT
)

if audit_directory:
    os.makedirs(
        audit_directory,
        exist_ok=True,
    )


identity_audit.to_csv(
    IDENTITY_AUDIT_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)


print("Identity enrichment summary:")

print(
    identity_audit[
        "MATCH_STATUS"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nIdentity audit written to:"
)

print(
    IDENTITY_AUDIT_OUTPUT
)


# ------------------------------------------------------------
# 18. Review rows + hard blocker gate
# ------------------------------------------------------------

# Corrected conflicts are warnings because athlete_master has
# supplied the authoritative final DOB/identity.
warning_statuses = {
    "UID_DOB_CONFLICT_CORRECTED",
    "NAME_MATCH_DOB_MISMATCH_CORRECTED",
    "TOKEN_ORDER_DOB_MISMATCH_CORRECTED",
    "MATCHED_BY_NAME_ONLY_DOB_UNAVAILABLE_IN_REFERENCE",
    "MATCHED_BY_TOKEN_ORDER_DOB_UNAVAILABLE_IN_REFERENCE",
}

# These statuses are not safe for production upload.
blocking_statuses = {
    "UNKNOWN_UNIQUE_ID",
    "MULTIPLE_UID_CANDIDATES",
    "MATCHED_BUT_NO_UID",
    "NO_MATCH",
}

warning_rows = identity_audit[
    identity_audit["MATCH_STATUS"].isin(
        warning_statuses
    )
].copy()

blocking_identity_rows = identity_audit[
    identity_audit["MATCH_STATUS"].isin(
        blocking_statuses
    )
].copy()

review_rows = identity_audit[
    identity_audit["MATCH_STATUS"].isin(
        warning_statuses | blocking_statuses
    )
].copy()

print(
    f"\nIdentity warning rows: "
    f"{len(warning_rows)}"
)

print(
    f"Blocking identity rows: "
    f"{len(blocking_identity_rows)}"
)

if not warning_rows.empty:
    print(
        "\nIdentity warnings corrected using authoritative "
        "reference data:"
    )
    display(warning_rows)

if not blocking_identity_rows.empty:
    print(
        "\nBLOCKING IDENTITY ERRORS:"
    )
    display(blocking_identity_rows)

    raise ValueError(
        "Identity enrichment produced blocking rows. "
        "Resolve these identities before BigQuery upload."
    )

print(
    "\nIdentity gate passed: no blocking identity rows."
)

# This flag is available for the final pre-upload gate in Cell 43.
IDENTITY_VALIDATED = True


# ------------------------------------------------------------
# 19. Validate Louis and Emir
# ------------------------------------------------------------

target_keys = {
    "louismarcbrian",
    "muhammadrashidemir",
}


target_audit = identity_audit[
    identity_audit[
        "NAME_MATCH_KEY"
    ].isin(target_keys)
].copy()


print(
    "\nLouis and Emir validation:"
)


if not target_audit.empty:
    display(
        target_audit[
            [
                "RAW_NAME",
                "STANDARDIZED_NAME",
                "FINAL_UNIQUE_ID",
                "RAW_DOB",
                "NORMALIZED_INCOMING_DOB",
                "CANDIDATE_DOBS",
                "FINAL_DOB",
                "MATCH_STATUS",
                "MATCH_REASON",
            ]
        ]
    )
else:
    print(
        "No Louis or Emir rows were found "
        "in the incoming dataframe."
    )


# ------------------------------------------------------------
# 20. Drop helper columns
# ------------------------------------------------------------

helper_columns = [
    "_NAME_MATCH_KEY",
    "_NAME_TOKEN_SIGNATURE",
    "_DOB_NORMALIZED",
    "_RAW_UNIQUE_ID",
    "_RAW_NAME",
    "_RAW_DOB",
    "_MATCH_STATUS",
    "_MATCH_SOURCE",
    "_MATCH_REASON",
    "_CANDIDATE_UNIQUE_IDS",
    "_CANDIDATE_DOBS",
    "_CANDIDATE_NAMES",
]


df.drop(
    columns=helper_columns,
    inplace=True,
    errors="ignore",
)


# ------------------------------------------------------------
# 21. Save enriched dataframe
# ------------------------------------------------------------

final_output_directory = os.path.dirname(
    FINAL_OUTPUT_PATH
)

if final_output_directory:
    os.makedirs(
        final_output_directory,
        exist_ok=True,
    )


df.to_csv(
    FINAL_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "\nEnriched output written to:"
)

print(
    FINAL_OUTPUT_PATH
)


# ------------------------------------------------------------
# 22. Final verification
# ------------------------------------------------------------
'''
final_keys = df["NAME"].apply(
    name_match_key
)


final_louis = df[
    final_keys == "louismarcbrian"
].copy()


final_emir = df[
    final_keys == "emirbinmuhammadrashid"
].copy()


verification_columns = [
    column
    for column in [
        "NAME",
        "UNIQUE_ID",
        "DOB",
        "EVENT",
        "RESULT",
    ]
    if column in df.columns
]


print("\nFinal Louis rows:")

display(
    final_louis[
        verification_columns
    ]
)


print("\nFinal Emir rows:")

display(
    final_emir[
        verification_columns
    ]
)

'''

Identity enrichment summary:
MATCH_STATUS
NO_MATCH               21
MATCHED_BY_NAME_DOB    11
Name: count, dtype: int64

Identity audit written to:
/Users/veesheenyuen/Desktop/DataScience/SAA/External events/2026/Aug/General/audit/identity_enrichment_audit.csv

Identity warning rows: 0
Blocking identity rows: 21

BLOCKING IDENTITY ERRORS:


,ROW_INDEX,RAW_NAME,STANDARDIZED_NAME,RAW_UNIQUE_ID,FINAL_UNIQUE_ID,RAW_DOB,NORMALIZED_INCOMING_DOB,FINAL_DOB,NAME_MATCH_KEY,NAME_TOKEN_SIGNATURE,MATCH_STATUS,MATCH_SOURCE,MATCH_REASON,CANDIDATE_UNIQUE_IDS,CANDIDATE_DOBS,CANDIDATE_NAMES,COMPETITION,EVENT,DATE,RESULT
0,0,Yao Peng Lim,Yao Peng Lim,,,26 Feb 89,1989-02-26,26/02/1989,yaopenglim,lim|peng|yao,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,13.44
1,1,Weiyi Lu,Weiyi Lu,,,20 Dec 87,1987-12-20,20/12/1987,weiyilu,lu|weiyi,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,11.90
2,2,Shawn Wee,Shawn Wee,,,29 Apr 90,1990-04-29,29/04/1990,shawnwee,shawn|wee,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,11.82
3,3,Danny Lum,Danny Lum,,,13 Mar 82,1982-03-13,13/03/1982,dannylum,danny|lum,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,12.18
4,4,Curtis Liau,Curtis Liau,,,10 Sep 84,1984-09-10,10/09/1984,curtisliau,curtis|liau,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,12.02
5,5,Wei De Toh,Wei De Toh,,,27 Apr 86,1986-04-27,27/04/1986,weidetoh,de|toh|wei,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,Discus Throw,2026-08-22 00:00:00+00:00,37.26
6,6,Furene Wang,Furene Wang,,,27 Jul 84,1984-07-27,27/07/1984,furenewang,furene|wang,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-24 00:00:00+00:00,15.40
7,7,Anna Liisa Milani,Anna Liisa Milani,,,10 Sep 81,1981-09-10,10/09/1981,annaliisamilani,anna|liisa|milani,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-24 00:00:00+00:00,15.68
8,8,Aaron Augustine Bing Kai Huang,Aaron Augustine Bing Kai Huang,,,28 Oct 80,1980-10-28,28/10/1980,aaronaugustinebingkaihuang,aaron|augustine|bing|huang|kai,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,100m,2026-08-22 00:00:00+00:00,13.77
9,9,Zilin Jiang,Zilin Jiang,,,8 Jun 81,1981-06-08,08/06/1981,zilinjiang,jiang|zilin,NO_MATCH,,No exact name variation/canonical-name match a...,,,,26th World Masters Athletics Championships,High Jump,2026-08-23 00:00:00+00:00,1.50


ValueError: Identity enrichment produced blocking rows. Resolve these identities before BigQuery upload.

In [138]:
df

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,_RAW_DOB,_NAME_MATCH_KEY,_NAME_TOKEN_SIGNATURE,_DOB_NORMALIZED,_MATCH_STATUS,_MATCH_SOURCE,_MATCH_REASON,_CANDIDATE_UNIQUE_IDS,_CANDIDATE_DOBS,_CANDIDATE_NAMES
0,nan,nan,nan,Yao Peng Lim,7,nan,nan,nan,13.44,nan,...,26 Feb 89,yaopenglim,lim|peng|yao,1989-02-26,NO_MATCH,,No exact name variation/canonical-name match a...,,,
1,nan,nan,nan,Weiyi Lu,5,nan,nan,nan,11.90,nan,...,20 Dec 87,weiyilu,lu|weiyi,1987-12-20,NO_MATCH,,No exact name variation/canonical-name match a...,,,
2,nan,nan,nan,Shawn Wee,5,nan,nan,nan,11.82,nan,...,29 Apr 90,shawnwee,shawn|wee,1990-04-29,NO_MATCH,,No exact name variation/canonical-name match a...,,,
3,nan,nan,nan,Danny Lum,4,nan,nan,nan,12.18,nan,...,13 Mar 82,dannylum,danny|lum,1982-03-13,NO_MATCH,,No exact name variation/canonical-name match a...,,,
4,nan,nan,nan,Curtis Liau,4,nan,nan,nan,12.02,nan,...,10 Sep 84,curtisliau,curtis|liau,1984-09-10,NO_MATCH,,No exact name variation/canonical-name match a...,,,
5,nan,nan,nan,Wei De Toh,6,nan,nan,nan,37.26,nan,...,27 Apr 86,weidetoh,de|toh|wei,1986-04-27,NO_MATCH,,No exact name variation/canonical-name match a...,,,
6,nan,nan,nan,Furene Wang,6,nan,nan,nan,15.40,nan,...,27 Jul 84,furenewang,furene|wang,1984-07-27,NO_MATCH,,No exact name variation/canonical-name match a...,,,
7,nan,nan,nan,Anna Liisa Milani,6,nan,nan,nan,15.68,nan,...,10 Sep 81,annaliisamilani,anna|liisa|milani,1981-09-10,NO_MATCH,,No exact name variation/canonical-name match a...,,,
8,nan,nan,nan,Aaron Augustine Bing Kai Huang,6,nan,nan,nan,13.77,nan,...,28 Oct 80,aaronaugustinebingkaihuang,aaron|augustine|bing|huang|kai,1980-10-28,NO_MATCH,,No exact name variation/canonical-name match a...,,,
9,nan,nan,nan,Zilin Jiang,3,nan,nan,nan,1.50,nan,...,8 Jun 81,zilinjiang,jiang|zilin,1981-06-08,NO_MATCH,,No exact name variation/canonical-name match a...,,,


In [139]:
'''
gender_pattern = 'Male|Female|Mixed'

date_year = pd.to_datetime(
    row['DATE'],
    errors='coerce',
    utc=True
).year

format_string = "%Y-%m-%d %H:%M:%S%z"

stage_list = ['Final', 'Semifinal', 'Heats', 'Qualification', 'Preliminaries', '']
event_list = [
    '60m', '60m Hurdles', '100m', '300m', '400m', '800m', '1000m', '1500m', '1500m Walk', 
    '3000m', '5000m', '10,000m', '80m Hurdles', '110m Hurdles', '100m Hurdles', 
    '200m Hurdles', '300m Hurdles', '400m Hurdles', 'High Jump', 'Discus', 
    'Discus Throw', 'Triple Jump', 'Pole Vault', 'Long Jump', 'Javelin', 
    'Javelin Throw', 'Shot Put', 'High Jump', '2000m SC', '3000m Walk', '2000m Steeplechase',
    '5000m Walk', 'Hammer Throw', '200m', '3000m Steeplechase', '4 x 100m', 'Trail', 
    '4 x 400m', '1500m Racewalk', '3000m Racewalk', '5000m Racewalk', 'Mile Road', '10km Road',
    'Half Marathon', 'Marathon', 'Cross Country 10 km', 'Mountain', 'Mountain Running',
    'Half Marathon Racewalk', '10km Racewalk', '20km Racewalk', 'Heptathlon'
]
category_list = [
    'Throw', 'Jump', 'Relay', 'Long', 'Mid', 'Sprint', 'Road', 'Trail',
    'Cross Country', 'Steeple', 'Hurdles', 'Walk', 'Marathon', 'Cross Country',
    'Triathlon', 'Decathlon', 'Pentathlon', 'Octathlon', 'Mountain', 'Heptathlon'
]

valid_event_class = [
    '5m', 'Turbo (400g)', '0.838m', '5kg', '0.914m', '76.2 cm - 7m', '1.75', '1kg', 'Cross Country', '4kg', '7.26kg',
    '1.5kg', '0.991m', '1kg Medicine', '6kg', '0.60m', '0.68m', '500g', '0.686m', '0.991m', '1.067m', '10m', '0.50m', '800g', '700g', '2x2',
    '400g Turbo J', '400g', 'Standing', '2kg', '1.50kg', '0.840m', '300g', '0.44m', '0.84m', '3kg', '0.762m', 'Trial',
    '600g', '50cm', '2kg Med Ball', 'Scissor', '1.75kg', 'Turbo (300g)'
]

time_categories = {
    'Relay', 'Long', 'Mid', 'Sprint', 'Road', 'Trail',
    'Cross Country', 'Steeple', 'Hurdles', 'Walk',
    'Marathon', 'Mountain'
}

distance_categories = {'Throw', 'Jump'}

middle_distance_events = {'1500m', '3000m', '5000m', '10,000m'}
marathon_events = {'Half Marathon', 'Marathon'}

special_result_codes = {'NM', 'DNF', 'DQ', 'DNS'}

format_string = "%Y-%m-%d %H:%M:%S%z"

# ----------------------------
# Centralized cleaning step
# ----------------------------
# Convert to string where necessary
df['YEAR'] = df['YEAR'].astype(str)
df['DATE'] = df['DATE'].astype(str)
df['DOB'] = df['DOB'].astype(str)

# Normalize all missing/null values (empty, None, nan, NaN) → pd.NA
df = df.replace(["", "None", "none", "nan", "NaN"], pd.NA)

# Quick check of missing values
missing_summary = df.isna().sum()
print("Missing values per column:\n", missing_summary)

# Events that must have a wind value

wind_required_events = {
    '100m',
    '200m',
    '100m Hurdles',
    '110m Hurdles',
    'Triple Jump',
    'Long Jump'
}


def is_valid_wind(value):
    # Missing is invalid for wind-required events
    if pd.isna(value):
        return False

    # Accept numeric types directly
    if isinstance(value, (int, float)) and not pd.isna(value):
        return True

    # Accept strings like "0.3", "-1.2", "+0.4", or "NWI"
    value = str(value).strip()
    if value.upper() == "NWI":
        return True

    try:
        float(value)
        return True
    except ValueError:
        return False


def is_valid_result_value(value): # checks for valid float or time value
    """
    Accepts:
    - SS.ss          -> 10.52
    - M:SS.ss        -> 1:52.34
    - H:MM:SS.ss     -> 2:15:09.45
    - whole numbers  -> 52
    - special codes  -> NM, DNF, DQ, DNS
    """
    if pd.isna(value):
        return False

    value = str(value).strip().upper()

    if value in special_result_codes:
        return True

    patterns = [
        r'^\d+(\.\d+)?W?$',               # 10, 10.52, 7.00W
        r'^\d{1,2}:\d{2}(\.\d+)?$',       # 1:52 or 1:52.34
        r'^\d{1,2}:\d{2}:\d{2}(\.\d+)?$'  # 2:15:09 or 2:15:09.45
    ]

    return any(re.fullmatch(pattern, value) for pattern in patterns)


def is_valid_result(row): # checks for valid float or time value
    event = str(row['EVENT']).strip()
    category = str(row['CATEGORY_EVENT']).strip()
    result = str(row['RESULT']).strip()
    result_upper = result.upper()

    # Allow special codes everywhere
    if result_upper in special_result_codes:
        return True

    # Specific long-distance track events:
    # allow MM:SS.ss or 00:MM:SS.ss
    if event in middle_distance_events:
        return bool(
            re.fullmatch(r'^\d{1,2}:\d{2}(\.\d+)?$', result) or
            re.fullmatch(r'^00:\d{1,2}:\d{2}(\.\d+)?$', result)
        )

    # Marathon events:
    # require HH:MM:SS or HH:MM:SS.ss
    if event in marathon_events:
        return bool(
            re.fullmatch(r'^\d{1,2}:\d{2}:\d{2}(\.\d+)?$', result)
        )

    # Field events: must be numeric only, not colon-formatted time
    if category in distance_categories:
        return is_valid_result_value(result) and ':' not in result

    # Other timed events
    if category in time_categories:
        return is_valid_result_value(result)

    # Skip categories like Heptathlon unless you want separate rules
    return True

def is_valid_date_format(value, fmt="%Y-%m-%d %H:%M:%S%z"):
    if pd.isna(value):
        return False
    try:
        datetime.datetime.strptime(str(value).strip(), fmt)
        return True
    except ValueError:
        return False

# ----------------------------
# Validation loop
# ----------------------------
for index, row in df.iterrows():
    # Generic null checks
    assert not pd.isna(row['NAME']), f"NAME missing at row {index}"
    assert not pd.isna(row['DATE']), f"DATE missing at row {index}"
    assert not pd.isna(row['EVENT']), f"EVENT missing at row {index}"
    assert not pd.isna(row['COMPETITION']), f"COMPETITION missing at row {index}"

    # Domain-specific validations
#    assert re.search(gender_pattern, row['GENDER']) is not None, f"Invalid GENDER at row {index}"
#    assert row['REGION'] in ['Local', 'International'], f"Invalid REGION at row {index}"
#    assert row['CATEGORY_EVENT'] in category_list, f"Invalid CATEGORY_EVENT at row {index}"
#    assert re.search(r'\)|\(', row['EVENT']) is None, f"Invalid EVENT (brackets) at row {index}"
#    assert row['NATIONALITY'] == 'SGP', f"Unexpected NATIONALITY at row {index}"
#    assert re.search(year_pattern, row['YEAR']) is not None, f"Invalid YEAR at row {index}"
#    assert row['STAGE'] in stage_list, f"Invalid STAGE at row {index}"
#    assert row['EVENT'] in event_list, f"Invalid EVENT at row {index}"

# Domain-specific validations with value reporting
    assert re.search(gender_pattern, row['GENDER']) is not None, (f"Invalid GENDER '{row['GENDER']}' at row {index}")

    assert row['REGION'] in ['Local', 'International'], (f"Invalid REGION '{row['REGION']}' at row {index}")

    assert row['CATEGORY_EVENT'] in category_list, (f"Invalid CATEGORY_EVENT '{row['CATEGORY_EVENT']}' at row {index}")

    assert re.search(r'\)|\(', row['EVENT']) is None, (f"Invalid EVENT (brackets) '{row['EVENT']}' at row {index}")

   # assert row['NATIONALITY'] == 'SGP', (f"Unexpected NATIONALITY '{row['NATIONALITY']}' at row {index}")
    assert str(row['NATIONALITY']).upper() != 'SIN', (f"Invalid NATIONALITY '{row['NATIONALITY']}' at row {index}") # check that only SGP exists

    assert int(row['YEAR']) == date_year, (
        f"YEAR '{row['YEAR']}' does not match "
        f"DATE '{row['DATE']}' at row {index}"
    )

    assert row['EVENT'] in event_list, (f"Invalid EVENT '{row['EVENT']}' at row {index}")

    assert is_valid_result(row), (f"Invalid RESULT '{row['RESULT']}' for EVENT '{row['EVENT']}' "
    f"with CATEGORY_EVENT '{row['CATEGORY_EVENT']}' at row {index}"
    )

    
    # Wind check for applicable events
  #  if row['EVENT'] in wind_required_events:
  #      assert is_valid_wind(row['WIND']), (
  #          f"Invalid or missing WIND '{row['WIND']}' for EVENT '{row['EVENT']}' at row {index}"
  #      )

    # ---- Stage check (allow NaN / None / null)
    if pd.notna(row['STAGE']) and row['STAGE'] not in stage_list:
        raise ValueError(f"Invalid STAGE: {row['STAGE']}")

    # ---- Event Class check (allow NaN / None / null)
    if pd.notna(row['EVENT_CLASS']) and row['EVENT_CLASS'] not in valid_event_class:
        raise ValueError(f"Invalid EVENT CLASS: {row['EVENT_CLASS']}")

    
    # Date parsing check
  #  datetime.datetime.strptime(row['DATE'], format_string)
    assert is_valid_date_format(row['DATE'], format_string), (
    f"Invalid DATE '{row['DATE']}' at row {index}; "
    f"expected format like '2026-04-17 00:00:00+00:00'"
    )

    
# Column structure validation
assert len(df.columns) == 40, "Unexpected number of columns"
assert df.columns.get_loc('RESULT') == 8, "RESULT column misplaced"
assert df.columns.get_loc('AGE') == 17, "AGE column misplaced"
assert df.columns.get_loc('WIND') == 12, "WIND column misplaced"
assert df.columns.get_loc('COMPETITION') == 24, "COMPETITION column misplaced"

print("✅ ALL CHECKS PASSED")

'''

'\ngender_pattern = \'Male|Female|Mixed\'\n\ndate_year = pd.to_datetime(\n    row[\'DATE\'],\n    errors=\'coerce\',\n    utc=True\n).year\n\nformat_string = "%Y-%m-%d %H:%M:%S%z"\n\nstage_list = [\'Final\', \'Semifinal\', \'Heats\', \'Qualification\', \'Preliminaries\', \'\']\nevent_list = [\n    \'60m\', \'60m Hurdles\', \'100m\', \'300m\', \'400m\', \'800m\', \'1000m\', \'1500m\', \'1500m Walk\', \n    \'3000m\', \'5000m\', \'10,000m\', \'80m Hurdles\', \'110m Hurdles\', \'100m Hurdles\', \n    \'200m Hurdles\', \'300m Hurdles\', \'400m Hurdles\', \'High Jump\', \'Discus\', \n    \'Discus Throw\', \'Triple Jump\', \'Pole Vault\', \'Long Jump\', \'Javelin\', \n    \'Javelin Throw\', \'Shot Put\', \'High Jump\', \'2000m SC\', \'3000m Walk\', \'2000m Steeplechase\',\n    \'5000m Walk\', \'Hammer Throw\', \'200m\', \'3000m Steeplechase\', \'4 x 100m\', \'Trail\', \n    \'4 x 400m\', \'1500m Racewalk\', \'3000m Racewalk\', \'5000m Racewalk\', \'Mile Road\', \'10km Road\',\n    \'Half Mar

In [141]:
gender_pattern = 'Male|Female|Mixed'
    
format_string = "%Y-%m-%d %H:%M:%S%z"

stage_list = ['Final', 'Semifinal', 'Heats', 'Qualification', 'Preliminaries', '']
event_list = [
    '60m', '60m Hurdles', '100m', '300m', '400m', '800m', '1000m', '1500m', '2000m', '1500m Walk', 
    '3000m', '5000m', '10,000m', '80m Hurdles', '110m Hurdles', '100m Hurdles', 
    '200m Hurdles', '300m Hurdles', '400m Hurdles', 'High Jump', 'Discus', 
    'Discus Throw', 'Triple Jump', 'Pole Vault', 'Long Jump', 'Javelin', 'Medley Relay',
    'Javelin Throw', 'Shot Put', 'High Jump', '2000m SC', '3000m Walk', '2000m Steeplechase',
    '5000m Walk', 'Hammer Throw', '200m', '3000m Steeplechase', '4 x 100m', 'Trail', 
    '4 x 400m', '1500m Racewalk', '3000m Racewalk', '5000m Racewalk', 'Mile Road',
    'Half Marathon', 'Marathon', 'Cross Country 10 km', 'Mountain', 'Mountain Running', '10km Road',
    'Half Marathon Racewalk', '20km Racewalk', 'Heptathlon', 'Decathlon', 'Octathlon', 'Pentathlon'
]
category_list = [
    'Throw', 'Jump', 'Relay', 'Long', 'Mid', 'Sprint', 'Road', 'Trail',
    'Cross Country', 'Steeple', 'Hurdles', 'Walk', 'Marathon', 'Cross Country',
    'Triathlon', 'Decathlon', 'Pentathlon', 'Octathlon', 'Mountain', 'Heptathlon'
]

valid_event_class = [
    '5m', 'Turbo (400g)', '0.838m', '5kg', '0.914m', '76.2 cm - 7m', '1.75', '1kg', 'Cross Country', '4kg', '7.26kg',
    '1.5kg', '0.991m', '1kg Medicine', '6kg', '0.60m', '0.68m', '500g', '0.686m', '1.067m', '10m', '0.50m', '800g', '700g', '2x2',
    '400g Turbo J', '400g', 'Standing', '2kg', '1.50kg', '0.840m', '300g', '0.44m', '0.84m', '3kg', '0.762m', 'Trial',
    '600g', '50cm', '2kg Med Ball', 'Scissor', '1.75kg', 'Turbo (300g)'
]

time_categories = {
    'Relay', 'Long', 'Mid', 'Sprint', 'Road', 'Trail',
    'Cross Country', 'Steeple', 'Hurdles', 'Walk',
    'Marathon', 'Mountain'
}

distance_categories = {'Throw', 'Jump'}

middle_distance_events = {'1500m', '3000m', '5000m', '10,000m'}
marathon_events = {'Half Marathon', 'Marathon'}

special_result_codes = {'NM', 'DNF', 'DQ', 'DNS', 'FOUL', 'NH'}

format_string = "%Y-%m-%d %H:%M:%S%z"

# ----------------------------
# Centralized cleaning step
# ----------------------------
# Convert to string where necessary
df['YEAR'] = df['YEAR'].astype(str)
df['DATE'] = df['DATE'].astype(str)
df['DOB'] = df['DOB'].astype(str)

# Normalize all missing/null values (empty, None, nan, NaN) → pd.NA
df = df.replace(["", "None", "none", "nan", "NaN"], pd.NA)

# Quick check of missing values
missing_summary = df.isna().sum()
print("Missing values per column:\n", missing_summary)

# Events that must have a wind value

wind_required_events = {
    '100m',
    '200m',
    '100m Hurdles',
    '110m Hurdles',
    'Triple Jump',
    'Long Jump'
}


def is_valid_wind(value):

    if pd.isna(value):
        return False

    value = str(value).strip()

    if value == "":
        return False

    if value.casefold() in {
        "nwi",
        "illegal",
        "illegal wind",
    }:
        return True

    try:
        float(
            value.replace("+", "")
        )
        return True

    except ValueError:
        return False

def is_valid_result_value(value): # checks for valid float or time value
    """
    Accepts:
    - SS.ss          -> 10.52
    - M:SS.ss        -> 1:52.34
    - H:MM:SS.ss     -> 2:15:09.45
    - whole numbers  -> 52
    - special codes  -> NM, DNF, DQ, DNS
    """
    if pd.isna(value):
        return False

    value = str(value).strip().upper()

    if value in special_result_codes:
        return True

    patterns = [
    r'^\d+(?:\.\d+)?[MW]?$',
    r'^\d{1,2}:\d{2}(?:\.\d+)?$',
    r'^\d{1,2}:\d{2}:\d{2}(?:\.\d+)?$'
    ]
    
    return any(re.fullmatch(pattern, value) for pattern in patterns)


def is_valid_result(row):
    event = str(row['EVENT']).strip()
    category = str(row['CATEGORY_EVENT']).strip()
    result = str(row['RESULT']).strip()
    result_upper = result.upper()

    # Allow special codes everywhere
    if result_upper in special_result_codes:
        return True

    # Long-distance track events:
    # Accept MM:SS.ss or HH:MM:SS.ss
    if event in middle_distance_events:
        return bool(
            re.fullmatch(
                r'^(?:\d{1,2}:\d{2}(?:\.\d+)?|\d{1,2}:\d{2}:\d{2}(?:\.\d+)?)$',
                result
            )
        )

    # Marathon events:
    # Require HH:MM:SS or HH:MM:SS.ss
    if event in marathon_events:
        return bool(
            re.fullmatch(r'^\d{1,2}:\d{2}:\d{2}(?:\.\d+)?$', result)
        )

    # Field events: must be numeric only
    if category in distance_categories:
        return is_valid_result_value(result) and ':' not in result

    # Other timed events
    if category in time_categories:
        return is_valid_result_value(result)

    return True


def is_valid_date_format(value, fmt="%Y-%m-%d %H:%M:%S%z"):
    if pd.isna(value):
        return False
    try:
        datetime.datetime.strptime(str(value).strip(), fmt)
        return True
    except ValueError:
        return False

# ----------------------------
# Validation loop
# ----------------------------
for index, row in df.iterrows():
    # Generic null checks
    assert not pd.isna(row['NAME']), f"NAME missing at row {index}"
    assert not pd.isna(row['DATE']), f"DATE missing at row {index}"
    assert not pd.isna(row['EVENT']), f"EVENT missing at row {index}"
    assert not pd.isna(row['COMPETITION']), f"COMPETITION missing at row {index}"

    # Domain-specific validations
#    assert re.search(gender_pattern, row['GENDER']) is not None, f"Invalid GENDER at row {index}"
#    assert row['REGION'] in ['Local', 'International'], f"Invalid REGION at row {index}"
#    assert row['CATEGORY_EVENT'] in category_list, f"Invalid CATEGORY_EVENT at row {index}"
#    assert re.search(r'\)|\(', row['EVENT']) is None, f"Invalid EVENT (brackets) at row {index}"
#    assert row['NATIONALITY'] == 'SGP', f"Unexpected NATIONALITY at row {index}"
#    assert re.search(year_pattern, row['YEAR']) is not None, f"Invalid YEAR at row {index}"
#    assert row['STAGE'] in stage_list, f"Invalid STAGE at row {index}"
#    assert row['EVENT'] in event_list, f"Invalid EVENT at row {index}"

# Domain-specific validations with value reporting
    assert re.search(gender_pattern, row['GENDER']) is not None, (f"Invalid GENDER '{row['GENDER']}' at row {index}")

    assert row['REGION'] in ['Local', 'International'], (f"Invalid REGION '{row['REGION']}' at row {index}")

    assert row['CATEGORY_EVENT'] in category_list, (f"Invalid CATEGORY_EVENT '{row['CATEGORY_EVENT']}' at row {index}")

    assert re.search(r'\)|\(', row['EVENT']) is None, (f"Invalid EVENT (brackets) '{row['EVENT']}' at row {index}")

   # assert row['NATIONALITY'] == 'SGP', (f"Unexpected NATIONALITY '{row['NATIONALITY']}' at row {index}")
    assert str(row['NATIONALITY']).upper() != 'SIN', (f"Invalid NATIONALITY '{row['NATIONALITY']}' at row {index}") # check that SGP is not shown as SIN

    parsed_date = pd.to_datetime(
    row['DATE'],
    errors='coerce',
    utc=True
    )
    
    assert pd.notna(parsed_date), (
        f"Invalid DATE '{row['DATE']}' at row {index}"
    )
    
    date_year = parsed_date.year

    assert int(row['YEAR']) == date_year, (
    f"YEAR '{row['YEAR']}' does not match "
    f"DATE '{row['DATE']}' at row {index}"
    )

    assert row['EVENT'] in event_list, (f"Invalid EVENT '{row['EVENT']}' at row {index}")

    assert is_valid_result(row), (f"Invalid RESULT '{row['RESULT']}' for EVENT '{row['EVENT']}' "
    f"with CATEGORY_EVENT '{row['CATEGORY_EVENT']}' at row {index}"
    )

    
    # Wind check for applicable events
    if row['EVENT'] in wind_required_events:
        assert is_valid_wind(row['WIND']), (
            f"Invalid or missing WIND '{row['WIND']}' for EVENT '{row['EVENT']}' at row {index}"
        )

    # ---- Stage check (allow NaN / None / null)
    if pd.notna(row['STAGE']) and row['STAGE'] not in stage_list:
        raise ValueError(f"Invalid STAGE: {row['STAGE']}")

    # ---- Event Class check (allow NaN / None / null)
    if pd.notna(row['EVENT_CLASS']) and row['EVENT_CLASS'] not in valid_event_class:
        raise ValueError(f"Invalid EVENT CLASS: {row['EVENT_CLASS']}")

    
    # Date parsing check
  #  datetime.datetime.strptime(row['DATE'], format_string)
    assert is_valid_date_format(row['DATE'], format_string), (
    f"Invalid DATE '{row['DATE']}' at row {index}; "
    f"expected format like '2026-04-17 00:00:00+00:00'"
    )

    
# Column structure validation
assert len(df.columns) == 40, "Unexpected number of columns"
assert df.columns.get_loc('RESULT') == 8, "RESULT column misplaced"
assert df.columns.get_loc('AGE') == 17, "AGE column misplaced"
assert df.columns.get_loc('WIND') == 12, "WIND column misplaced"
assert df.columns.get_loc('COMPETITION') == 24, "COMPETITION column misplaced"


DOMAIN_VALIDATED = True

print("✅ ALL CHECKS PASSED")

Missing values per column:
 FIRST_NAME               32
LAST_NAME                32
OTHER_NAME               32
NAME                      0
RANK                      0
TAG_ID                   32
TEAM                     32
SEED                     32
RESULT                    0
QUALIFICATION            31
HEAT                     16
LANE                     32
WIND                     10
EVENT                     0
DIVISION                 32
STAGE                    20
POINTS                   32
AGE                      32
GENDER                    0
UNIQUE_ID                21
NATIONALITY               0
DICT_RESULTS              0
YEAR                      0
DATE                      0
COMPETITION               0
REGION                    0
DOB                       0
GROUP                    32
CATEGORY_EVENT            0
ATHLETE_ID               32
SOURCE                    0
REMARKS                   0
TIMESTAMP                32
VENUE                     0
SUB_EVENT           

AssertionError: Invalid EVENT '80m Hurdles, 68 cm' at row 13

In [143]:
# Set timestamp

import datetime


timezone = pytz.timezone('UTC')
dt_ms = datetime.datetime.now().replace(tzinfo=timezone)
df.loc[:,'TIMESTAMP'] = dt_ms.replace(second=0, microsecond=0)


In [144]:
df

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,_RAW_DOB,_NAME_MATCH_KEY,_NAME_TOKEN_SIGNATURE,_DOB_NORMALIZED,_MATCH_STATUS,_MATCH_SOURCE,_MATCH_REASON,_CANDIDATE_UNIQUE_IDS,_CANDIDATE_DOBS,_CANDIDATE_NAMES
0,<NA>,<NA>,<NA>,Yao Peng Lim,7,<NA>,<NA>,<NA>,13.44,<NA>,...,26 Feb 89,yaopenglim,lim|peng|yao,1989-02-26,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,Weiyi Lu,5,<NA>,<NA>,<NA>,11.90,<NA>,...,20 Dec 87,weiyilu,lu|weiyi,1987-12-20,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,Shawn Wee,5,<NA>,<NA>,<NA>,11.82,<NA>,...,29 Apr 90,shawnwee,shawn|wee,1990-04-29,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,Danny Lum,4,<NA>,<NA>,<NA>,12.18,<NA>,...,13 Mar 82,dannylum,danny|lum,1982-03-13,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,Curtis Liau,4,<NA>,<NA>,<NA>,12.02,<NA>,...,10 Sep 84,curtisliau,curtis|liau,1984-09-10,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,Wei De Toh,6,<NA>,<NA>,<NA>,37.26,<NA>,...,27 Apr 86,weidetoh,de|toh|wei,1986-04-27,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,Furene Wang,6,<NA>,<NA>,<NA>,15.40,<NA>,...,27 Jul 84,furenewang,furene|wang,1984-07-27,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,Anna Liisa Milani,6,<NA>,<NA>,<NA>,15.68,<NA>,...,10 Sep 81,annaliisamilani,anna|liisa|milani,1981-09-10,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,Aaron Augustine Bing Kai Huang,6,<NA>,<NA>,<NA>,13.77,<NA>,...,28 Oct 80,aaronaugustinebingkaihuang,aaron|augustine|bing|huang|kai,1980-10-28,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,Zilin Jiang,3,<NA>,<NA>,<NA>,1.50,<NA>,...,8 Jun 81,zilinjiang,jiang|zilin,1981-06-08,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>


In [145]:
df['DATE'] = pd.to_datetime(df['DATE'], format='mixed', dayfirst=False, utc=True)
df['DATE'] = df['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64

df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='mixed', dayfirst=False, utc=True)
df['TIMESTAMP'] = df['TIMESTAMP'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64

In [146]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug')

df.to_excel("final_check.xlsx", index=False)

In [147]:
from google.cloud import bigquery
from google.cloud.bigquery import SchemaField

client = bigquery.Client()

# Load BQ Modified Schema for TEST

modified_schema = [
            bigquery.SchemaField("FIRST_NAME", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("LAST_NAME", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("OTHER_NAME", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("NAME", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("RANK", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("TAG_ID", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("TEAM", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("SEED", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("RESULT", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("QUALIFICATION", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("HEAT", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("LANE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("WIND", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("EVENT", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("DIVISION", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("STAGE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("POINTS", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("AGE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("GENDER", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("UNIQUE_ID", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("NATIONALITY", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("DICT_RESULTS", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("YEAR", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("DATE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("COMPETITION", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("REGION", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("DOB", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("GROUP", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("CATEGORY_EVENT", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("ATHLETE_ID", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("SOURCE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("REMARKS", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("TIMESTAMP", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("VENUE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("SUB_EVENT", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("SESSION", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("EVENT_CLASS", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("DISTANCE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("HOST_CITY", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("RX_TIME", "STRING", mode="NULLABLE")


]

In [148]:
df

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,_RAW_DOB,_NAME_MATCH_KEY,_NAME_TOKEN_SIGNATURE,_DOB_NORMALIZED,_MATCH_STATUS,_MATCH_SOURCE,_MATCH_REASON,_CANDIDATE_UNIQUE_IDS,_CANDIDATE_DOBS,_CANDIDATE_NAMES
0,<NA>,<NA>,<NA>,Yao Peng Lim,7,<NA>,<NA>,<NA>,13.44,<NA>,...,26 Feb 89,yaopenglim,lim|peng|yao,1989-02-26,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,Weiyi Lu,5,<NA>,<NA>,<NA>,11.90,<NA>,...,20 Dec 87,weiyilu,lu|weiyi,1987-12-20,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,Shawn Wee,5,<NA>,<NA>,<NA>,11.82,<NA>,...,29 Apr 90,shawnwee,shawn|wee,1990-04-29,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,Danny Lum,4,<NA>,<NA>,<NA>,12.18,<NA>,...,13 Mar 82,dannylum,danny|lum,1982-03-13,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,Curtis Liau,4,<NA>,<NA>,<NA>,12.02,<NA>,...,10 Sep 84,curtisliau,curtis|liau,1984-09-10,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,Wei De Toh,6,<NA>,<NA>,<NA>,37.26,<NA>,...,27 Apr 86,weidetoh,de|toh|wei,1986-04-27,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,Furene Wang,6,<NA>,<NA>,<NA>,15.40,<NA>,...,27 Jul 84,furenewang,furene|wang,1984-07-27,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,Anna Liisa Milani,6,<NA>,<NA>,<NA>,15.68,<NA>,...,10 Sep 81,annaliisamilani,anna|liisa|milani,1981-09-10,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,Aaron Augustine Bing Kai Huang,6,<NA>,<NA>,<NA>,13.77,<NA>,...,28 Oct 80,aaronaugustinebingkaihuang,aaron|augustine|bing|huang|kai,1980-10-28,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,Zilin Jiang,3,<NA>,<NA>,<NA>,1.50,<NA>,...,8 Jun 81,zilinjiang,jiang|zilin,1981-06-08,NO_MATCH,<NA>,No exact name variation/canonical-name match a...,<NA>,<NA>,<NA>


In [149]:
df = df.astype(str)

In [150]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2026/Aug/')

df.to_excel("check.xlsx", index=False)

In [151]:
# Ensure dataframe types match BigQuery modified_schema
# All fields in modified_schema are currently STRING

string_columns = [
    field.name
    for field in modified_schema
    if field.field_type == "STRING"
]

for col in string_columns:
    if col in df.columns:
        df[col] = df[col].astype("string")

# Verify no datetime columns remain
datetime_cols = df.select_dtypes(
    include=["datetime64[ns]", "datetimetz"]
).columns.tolist()

print("Datetime columns remaining:", datetime_cols)

assert not datetime_cols, (
    f"These columns are still datetime but BigQuery expects STRING: "
    f"{datetime_cols}"
)

df.info()

Datetime columns remaining: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 52 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   FIRST_NAME             32 non-null     string
 1   LAST_NAME              32 non-null     string
 2   OTHER_NAME             32 non-null     string
 3   NAME                   32 non-null     string
 4   RANK                   32 non-null     string
 5   TAG_ID                 32 non-null     string
 6   TEAM                   32 non-null     string
 7   SEED                   32 non-null     string
 8   RESULT                 32 non-null     string
 9   QUALIFICATION          32 non-null     string
 10  HEAT                   32 non-null     string
 11  LANE                   32 non-null     string
 12  WIND                   32 non-null     string
 13  EVENT                  32 non-null     string
 14  DIVISION               32 non-null     string

In [152]:
# ============================================================
# PRE-UPLOAD DUPLICATE PROTECTION
# ============================================================

from collections import Counter
import re

DUPLICATE_UPLOAD_VALIDATED = False
PREUPLOAD_VALIDATED = False
POST_UPLOAD_RECONCILED = False

PRODUCTION_TABLE = "saa-analytics.results.TEST"


# ------------------------------------------------------------
# 1. Assign one unique timestamp to this upload batch
# ------------------------------------------------------------

UPLOAD_BATCH_TIMESTAMP = pd.Timestamp.now(tz="UTC")

UPLOAD_BATCH_TIMESTAMP_TEXT = (
    UPLOAD_BATCH_TIMESTAMP.strftime(
        "%Y-%m-%d %H:%M:%S.%f+00:00"
    )
)

df["TIMESTAMP"] = UPLOAD_BATCH_TIMESTAMP_TEXT

print(
    "Upload batch timestamp:",
    UPLOAD_BATCH_TIMESTAMP_TEXT
)


# ------------------------------------------------------------
# 2. Fingerprint columns
# ------------------------------------------------------------
# Compare all production fields except TIMESTAMP.
# TIMESTAMP identifies the upload batch, not the athletics result.

# Fingerprint ONLY columns that actually belong to
# the BigQuery PRODUCTION schema.
#
# Do not allow temporary/helper dataframe columns such as
# _RAW_NAME, _MATCH_STATUS, etc. into the BigQuery query.

schema_cols = [
    field.name
    for field in modified_schema
]

FINGERPRINT_COLUMNS = [
    col
    for col in schema_cols
    if col != "TIMESTAMP"
]

NULL_TEXT_VALUES = {
    "",
    "nan",
    "none",
    "<na>",
    "nat",
    "null",
}


def normalise_fingerprint_value(column, value):

    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.casefold() in NULL_TEXT_VALUES:
        return ""

    # DOB uses the established SAA DOB normalisation logic.
    if column == "DOB":
    
        normalized_dob = normalize_incoming_dob(
            text
        )
    
        if normalized_dob:
            return normalized_dob
    
    
    # Competition DATE is already ISO/timezone-compatible.
    if column == "DATE":
    
        parsed = pd.to_datetime(
            text,
            errors="coerce",
            utc=True,
        )
    
        if pd.notna(parsed):
            return parsed.strftime("%Y-%m-%d")
    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.casefold()


def build_result_fingerprint(row):

    values = [
        normalise_fingerprint_value(
            column,
            row[column],
        )
        for column in FINGERPRINT_COLUMNS
    ]

    return "||".join(values)


df["_RESULT_FINGERPRINT"] = (
    df.apply(
        build_result_fingerprint,
        axis=1,
    )
)


# ------------------------------------------------------------
# 3. Detect exact duplicates inside candidate input
# ------------------------------------------------------------

input_duplicate_mask = (
    df["_RESULT_FINGERPRINT"]
    .duplicated(
        keep=False
    )
)

input_duplicates = (
    df.loc[
        input_duplicate_mask
    ]
    .copy()
)

if not input_duplicates.empty:

    print(
        "DUPLICATE ROWS FOUND INSIDE INPUT FILE:",
        len(input_duplicates)
    )

    display(
        input_duplicates[
            [
                col
                for col in [
                    "NAME",
                    "UNIQUE_ID",
                    "DATE",
                    "EVENT",
                    "COMPETITION",
                    "STAGE",
                    "HEAT",
                    "LANE",
                    "RESULT",
                    "EVENT_CLASS",
                ]
                if col in input_duplicates.columns
            ]
        ]
    )

    raise ValueError(
        "Upload blocked: exact duplicate result rows "
        "exist inside the candidate upload file."
    )

print(
    "✅ No exact duplicate rows inside input file."
)


# ------------------------------------------------------------
# 4. Limit BigQuery comparison to relevant competitions/years
# ------------------------------------------------------------

candidate_competitions = sorted({
    str(value).strip()
    for value in df["COMPETITION"]
    if str(value).strip()
    and str(value).strip().casefold()
    not in NULL_TEXT_VALUES
})

candidate_years = sorted({
    str(value).strip()
    for value in df["YEAR"]
    if str(value).strip()
    and str(value).strip().casefold()
    not in NULL_TEXT_VALUES
})


if not candidate_competitions:
    raise ValueError(
        "Cannot perform duplicate protection: "
        "no valid COMPETITION values found."
    )


if not candidate_years:
    raise ValueError(
        "Cannot perform duplicate protection: "
        "no valid YEAR values found."
    )


select_columns_sql = ",\n    ".join(
    f"`{column}`"
    for column in FINGERPRINT_COLUMNS
)


duplicate_check_sql = f"""
SELECT
    {select_columns_sql}
FROM `{PRODUCTION_TABLE}`
WHERE COMPETITION IN UNNEST(@competitions)
  AND CAST(YEAR AS STRING) IN UNNEST(@years)
"""


duplicate_check_job_config = (
    bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "competitions",
                "STRING",
                candidate_competitions,
            ),
            bigquery.ArrayQueryParameter(
                "years",
                "STRING",
                candidate_years,
            ),
        ]
    )
)


existing_bq_rows = (
    client.query(
        duplicate_check_sql,
        job_config=duplicate_check_job_config,
    )
    .to_dataframe()
)

print(
    "Potentially overlapping BQ rows checked:",
    len(existing_bq_rows)
)


# ------------------------------------------------------------
# 5. Compare candidate rows with existing PRODUCTION rows
# ------------------------------------------------------------

if not existing_bq_rows.empty:

    existing_bq_rows[
        "_RESULT_FINGERPRINT"
    ] = (
        existing_bq_rows.apply(
            build_result_fingerprint,
            axis=1,
        )
    )

    existing_fingerprints = set(
        existing_bq_rows[
            "_RESULT_FINGERPRINT"
        ]
    )

    already_uploaded_rows = (
        df.loc[
            df[
                "_RESULT_FINGERPRINT"
            ].isin(
                existing_fingerprints
            )
        ]
        .copy()
    )

else:

    already_uploaded_rows = pd.DataFrame()


if not already_uploaded_rows.empty:

    print(
        "ROWS ALREADY PRESENT IN BIGQUERY:",
        len(already_uploaded_rows)
    )

    display(
        already_uploaded_rows[
            [
                col
                for col in [
                    "NAME",
                    "UNIQUE_ID",
                    "DATE",
                    "EVENT",
                    "COMPETITION",
                    "STAGE",
                    "HEAT",
                    "LANE",
                    "RESULT",
                    "EVENT_CLASS",
                ]
                if col
                in already_uploaded_rows.columns
            ]
        ]
    )

    raise ValueError(
        "Upload blocked: one or more candidate rows "
        "already exist in PRODUCTION."
    )


print(
    "✅ No candidate rows already exist "
    "in BigQuery PRODUCTION."
)


# ------------------------------------------------------------
# 6. Preserve expected upload state
# ------------------------------------------------------------

EXPECTED_UPLOAD_ROWS = len(df)

EXPECTED_UPLOAD_FINGERPRINT_COUNTS = Counter(
    df["_RESULT_FINGERPRINT"]
)

DUPLICATE_UPLOAD_VALIDATED = True


print(
    "✅ DUPLICATE UPLOAD PROTECTION PASSED"
)

print(
    "Expected rows to upload:",
    EXPECTED_UPLOAD_ROWS
)

Upload batch timestamp: 2026-09-02 07:15:35.480240+00:00
✅ No exact duplicate rows inside input file.
Potentially overlapping BQ rows checked: 0
✅ No candidate rows already exist in BigQuery PRODUCTION.
✅ DUPLICATE UPLOAD PROTECTION PASSED
Expected rows to upload: 32


In [153]:
# ============================================================
# FINAL PRE-UPLOAD VALIDATION GATE
# ============================================================

PREUPLOAD_VALIDATED = False

required_validation_flags = {

    'SOURCE_SCHEMA_VALIDATED': globals().get(
        'SOURCE_SCHEMA_VALIDATED',
        False
    ),

    'DOMAIN_VALIDATED': globals().get(
        'DOMAIN_VALIDATED',
        False
    ),

    'IDENTITY_VALIDATED': globals().get(
        'IDENTITY_VALIDATED',
        False
    ),

    'DUPLICATE_UPLOAD_VALIDATED': globals().get(
        'DUPLICATE_UPLOAD_VALIDATED',
        False
    ),
}


failed_validation_stages = [
    flag
    for flag, passed
    in required_validation_flags.items()
    if not passed
]


if failed_validation_stages:

    raise RuntimeError(
        'PRE-UPLOAD VALIDATION FAILED. '
        'The following stages have not passed: '
        f'{failed_validation_stages}'
    )


PREUPLOAD_VALIDATED = True


print(
    '✅ PRE-UPLOAD VALIDATION COMPLETE — '
    'file is cleared for BigQuery upload.'
)

RuntimeError: PRE-UPLOAD VALIDATION FAILED. The following stages have not passed: ['IDENTITY_VALIDATED']

In [ ]:
# Write and append to PRODUCTION table in BigQuery directly with schema

POST_UPLOAD_RECONCILED = False

job_config = bigquery.LoadJobConfig(
    schema=modified_schema,
    write_disposition="WRITE_APPEND",  # or WRITE_APPEND, WRITE_EMPTY
)

if not globals().get(
    'PREUPLOAD_VALIDATED',
    False
):
    raise RuntimeError(
        'Upload blocked: '
        'PREUPLOAD_VALIDATED is not True.'
    )

# Confirm dataframe content has not changed since
# duplicate validation.

CURRENT_UPLOAD_FINGERPRINT_COUNTS = Counter(
    df.apply(
        build_result_fingerprint,
        axis=1
    )
)

if (
    CURRENT_UPLOAD_FINGERPRINT_COUNTS
    != EXPECTED_UPLOAD_FINGERPRINT_COUNTS
):
    raise RuntimeError(
        'Upload blocked: dataframe contents changed '
        'after duplicate validation. '
        'Rerun duplicate protection before uploading.'
    )


# Also confirm the upload batch timestamp has not changed.

if df['TIMESTAMP'].nunique(dropna=False) != 1:
    raise RuntimeError(
        'Upload blocked: more than one TIMESTAMP '
        'exists in the candidate batch.'
    )

if str(df['TIMESTAMP'].iloc[0]) != UPLOAD_BATCH_TIMESTAMP_TEXT:
    raise RuntimeError(
        'Upload blocked: batch TIMESTAMP changed '
        'after duplicate validation.'
    )

df_upload = (
    df.drop(
        columns=[
            '_RESULT_FINGERPRINT'
        ],
        errors='ignore',
    )
    .copy()
)


assert len(df_upload) == EXPECTED_UPLOAD_ROWS, (
    'Candidate row count changed after '
    'duplicate validation.'
)

#job = client.load_table_from_dataframe(df, 'saa-analytics.results.PRODUCTION', job_config=job_config)

upload_job = (
    client.load_table_from_dataframe(
        df_upload,
        'saa-analytics.results.TEST',
        job_config=job_config,
    )
)


upload_job.result()


print(
    '✅ BIGQUERY LOAD JOB COMPLETED'
)

print(
    'Rows submitted:',
    len(df_upload)
)

print(
    'Batch timestamp:',
    UPLOAD_BATCH_TIMESTAMP_TEXT
)



In [ ]:
# ============================================================
# POST-UPLOAD BIGQUERY RECONCILIATION
# ============================================================

from collections import Counter


if not globals().get(
    'PREUPLOAD_VALIDATED',
    False
):
    raise RuntimeError(
        'Cannot reconcile upload: '
        'PREUPLOAD_VALIDATED is not True.'
    )


if 'UPLOAD_BATCH_TIMESTAMP' not in globals():
    raise RuntimeError(
        'Cannot reconcile upload: '
        'UPLOAD_BATCH_TIMESTAMP is unavailable.'
    )


if (
    'EXPECTED_UPLOAD_FINGERPRINT_COUNTS'
    not in globals()
):
    raise RuntimeError(
        'Cannot reconcile upload: '
        'expected fingerprint state is unavailable.'
    )


# ------------------------------------------------------------
# 1. Retrieve exact batch just uploaded
# ------------------------------------------------------------

reconciliation_select_columns = (
    FINGERPRINT_COLUMNS
    + ['TIMESTAMP']
)


reconciliation_columns_sql = (
    ",\n    ".join(
        f"`{column}`"
        for column
        in reconciliation_select_columns
    )
)


reconciliation_sql = f"""
SELECT
    {reconciliation_columns_sql}
FROM `{PRODUCTION_TABLE}`
WHERE CAST(TIMESTAMP AS TIMESTAMP) = @batch_timestamp
"""


reconciliation_job_config = (
    bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                'batch_timestamp',
                'TIMESTAMP',
                UPLOAD_BATCH_TIMESTAMP.to_pydatetime(),
            )
        ]
    )
)


uploaded_batch = (
    client.query(
        reconciliation_sql,
        job_config=reconciliation_job_config,
    )
    .to_dataframe()
)


# ------------------------------------------------------------
# 2. Row-count reconciliation
# ------------------------------------------------------------

ACTUAL_UPLOAD_ROWS = len(
    uploaded_batch
)


print(
    'Expected upload rows:',
    EXPECTED_UPLOAD_ROWS
)

print(
    'Actual BigQuery rows:',
    ACTUAL_UPLOAD_ROWS
)


row_count_match = (
    ACTUAL_UPLOAD_ROWS
    == EXPECTED_UPLOAD_ROWS
)


# ------------------------------------------------------------
# 3. Rebuild fingerprints from returned BigQuery data
# ------------------------------------------------------------

if not uploaded_batch.empty:

    uploaded_batch[
        '_RESULT_FINGERPRINT'
    ] = (
        uploaded_batch.apply(
            build_result_fingerprint,
            axis=1,
        )
    )

else:

    uploaded_batch[
        '_RESULT_FINGERPRINT'
    ] = pd.Series(
        dtype='string'
    )


ACTUAL_UPLOAD_FINGERPRINT_COUNTS = Counter(
    uploaded_batch[
        '_RESULT_FINGERPRINT'
    ]
)


missing_fingerprints = (
    EXPECTED_UPLOAD_FINGERPRINT_COUNTS
    - ACTUAL_UPLOAD_FINGERPRINT_COUNTS
)


unexpected_fingerprints = (
    ACTUAL_UPLOAD_FINGERPRINT_COUNTS
    - EXPECTED_UPLOAD_FINGERPRINT_COUNTS
)


missing_count = sum(
    missing_fingerprints.values()
)

unexpected_count = sum(
    unexpected_fingerprints.values()
)


# ------------------------------------------------------------
# 4. Duplicate check within uploaded batch
# ------------------------------------------------------------

uploaded_duplicate_mask = (
    uploaded_batch[
        '_RESULT_FINGERPRINT'
    ]
    .duplicated(
        keep=False
    )
)


uploaded_duplicates = (
    uploaded_batch.loc[
        uploaded_duplicate_mask
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. Final reconciliation gate
# ------------------------------------------------------------

reconciliation_passed = (

    row_count_match

    and missing_count == 0

    and unexpected_count == 0

    and uploaded_duplicates.empty
)


if not reconciliation_passed:

    raise RuntimeError(
        'POST-UPLOAD RECONCILIATION FAILED. '
        f'Expected rows={EXPECTED_UPLOAD_ROWS}; '
        f'Actual rows={ACTUAL_UPLOAD_ROWS}; '
        f'Missing={missing_count}; '
        f'Unexpected={unexpected_count}; '
        f'Duplicate rows='
        f'{len(uploaded_duplicates)}.'
    )


POST_UPLOAD_RECONCILED = True


print(
    '✅ POST-UPLOAD RECONCILIATION PASSED'
)

print(
    f'Batch timestamp: '
    f'{UPLOAD_BATCH_TIMESTAMP_TEXT}'
)

print(
    f'Rows reconciled: '
    f'{ACTUAL_UPLOAD_ROWS}'
)

print(
    'Missing rows: 0'
)

print(
    'Unexpected rows: 0'
)

print(
    'Duplicate rows: 0'
)

In [49]:
'''
# Write and append to TEST table in BigQuery directly with schema

client = bigquery.Client()

job_config = bigquery.LoadJobConfig(
    schema=modified_schema,
    write_disposition="WRITE_APPEND",  # or WRITE_APPEND, WRITE_EMPTY
)

#job = client.load_table_from_dataframe(df, 'saa-analytics.results.PRODUCTION_CORRECT', job_config=job_config)
job = client.load_table_from_dataframe(df, 'saa-analytics.results.PRODUCTION', job_config=job_config)

job.result()  # Wait for the job to complete

'''

LoadJob<project=saa-analytics, location=US, id=e7da6440-a088-4036-8da4-91c28996c07a>

In [50]:
'''
# Check that loading was successful

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT *
FROM `saa-analytics.results.PRODUCTION`
WHERE TIMESTAMP='2026-08-30 15:47:00'

"""

extract = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)


extract

'''

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,SOURCE,REMARKS,TIMESTAMP,VENUE,SUB_EVENT,SESSION,EVENT_CLASS,DISTANCE,HOST_CITY,RX_TIME
0,<NA>,<NA>,<NA>,Yao Peng Lim,7,<NA>,<NA>,<NA>,13.44,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,SB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
1,<NA>,<NA>,<NA>,Weiyi Lu,5,<NA>,<NA>,<NA>,11.90,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
2,<NA>,<NA>,<NA>,Shawn Wee,5,<NA>,<NA>,<NA>,11.82,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
3,<NA>,<NA>,<NA>,Danny Lum,4,<NA>,<NA>,<NA>,12.18,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
4,<NA>,<NA>,<NA>,Curtis Liau,4,<NA>,<NA>,<NA>,12.02,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
5,<NA>,<NA>,<NA>,Wei De Toh,6,<NA>,<NA>,<NA>,37.26,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
6,<NA>,<NA>,<NA>,Furene Wang,6,<NA>,<NA>,<NA>,15.40,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
7,<NA>,<NA>,<NA>,Anna Liisa Milani,6,<NA>,<NA>,<NA>,15.68,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
8,<NA>,<NA>,<NA>,Aaron Augustine Bing Kai Huang,6,<NA>,<NA>,<NA>,13.77,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>
9,<NA>,<NA>,<NA>,Zilin Jiang,3,<NA>,<NA>,<NA>,1.50,<NA>,...,Tilastopaja - 26th World Masters Athletics Cha...,PB,2026-08-30 15:47:00,<NA>,Multievents,<NA>,<NA>,<NA>,Daegu,<NA>


In [110]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Tilastopaja/Singapore Athletes Events/2025/Sept/Batch 2')

df.to_excel("SGP_athletes_Sept_2025_consolidated_cleaned.xlsx")

# Read and process manually cleaned data

In [753]:
# read manually cleaned data

SGP_athletes = pd.read_csv("SGP_athletes_cleaned.csv")

In [754]:
SGP_athletes

,Unnamed: 0,RANK,NAME,NATIONALITY,RESULT,QUALIFICATION,COMPETITION,YEAR,DATE,EVENT,...,GENDER,STAGE,HEAT,WIND,DOB,REMARKS,RX_TIME,DICT_RESULT,SOURCE,REGION
0,102,4.0,Jun Jie Calvin Quek,SGP,53.19,NaN,UAE Athletics Grand Prix,2024,,400m Hurdles,...,Male,,,,26 Feb 96,« »,,[],Tilastopaja - UAE Athletics Grand Prix - May ...,International
1,31,2.0,Chui Ling Goh,SGP,17:33.73,NaN,11. Münchner Abendsportfest /MUC HochschulMS,2024,,5000m,...,Female,,,,27 Nov 92,« »,Mixed competition,[],Tilastopaja - 11. Münchner Abendsportfest :MU...,International
2,121,4.0,Chen Xiang Ang,SGP,8.01,NaN,11th Asian Indoor Athletics Championships,2024,17 February,60m Hurdles,...,Male,Heats,Heat 1,,3 Jul 94,SB «,,[],Tilastopaja - 11th Asian Indoor Athletics Cham...,International
3,112,8.0,Jun Jie Calvin Quek,SGP,52.05,NaN,11th Kinami Michitaka Memorial Athletics Meet,2024,,400m Hurdles,...,Male,,,,26 Feb 96,SB «,,[],Tilastopaja - 11th Kinami Michitaka Memorial A...,International
4,482,4.0,James Ethan Kai Meng Ang,SGP,52.55,NaN,11th Para Athletics World Championships,2024,22 May,400m,...,Male,Heats,Heat 2,,26 May 01,« »,0.187,[],Tilastopaja - 11th Para Athletics World Champi...,International
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,127,3.0,Jun Jie Calvin Quek,SGP,56.95,NaN,XXXIII Qosanov Memorial,2024,22 June,400m Hurdles,...,Male,Heats,Heat 2,,26 Feb 96,« »,,[],Tilastopaja - XXXIII Qosanov Memorial - Jun 24...,International
359,148,4.0,Kangli Emery Conrad,SGP,7.22,NaN,XXXIII Qosanov Memorial,2024,22 June,Long Jump,...,Male,NaN,NaN,NaN,26 Apr 06,« »,,[],Tilastopaja - XXXIII Qosanov Memorial - Jun 24...,International
360,320,5.0,Tia Louise Rozario,SGP,5.71,NaN,XXXIII Qosanov Memorial,2024,22 June,Long Jump,...,Female,NaN,NaN,NaN,14 Oct 00,« »,,[],Tilastopaja - XXXIII Qosanov Memorial - Jun 24...,International
361,330,5.0,Tia Louise Rozario,SGP,12.45,NaN,XXXIII Qosanov Memorial,2024,23 June,Triple Jump,...,Female,NaN,NaN,NaN,14 Oct 00,« »,,[],Tilastopaja - XXXIII Qosanov Memorial - Jun 24...,International


In [755]:
# Map general event category 

SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'100m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'400m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'600m', na=True), 'CATEGORY_EVENT'] = 'Mid'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'60m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'200m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'50m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'80m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'300m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'3000m', na=True), 'CATEGORY_EVENT'] = 'Long'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Mile', na=True), 'CATEGORY_EVENT'] = 'Mid'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'5000m', na=True), 'CATEGORY_EVENT'] = 'Long'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'2000m', na=True), 'CATEGORY_EVENT'] = 'Mid'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'800m', na=True), 'CATEGORY_EVENT'] = 'Mid'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'1500m', na=True), 'CATEGORY_EVENT'] = 'Mid'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'10,000m', na=True), 'CATEGORY_EVENT'] = 'Long'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'5km', na=True), 'CATEGORY_EVENT'] = 'Long'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'10km', na=True), 'CATEGORY_EVENT'] = 'Long'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Cross Country', na=True), 'CATEGORY_EVENT'] = 'Cross Country'


# Override values

SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Discus', na=True), 'CATEGORY_EVENT'] = 'Throw'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Javelin', na=True), 'CATEGORY_EVENT'] = 'Throw'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Shot', na=True), 'CATEGORY_EVENT'] = 'Throw'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Vault', na=True), 'CATEGORY_EVENT'] = 'Jump'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Throw', na=True), 'CATEGORY_EVENT'] = 'Throw'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Jump', na=True), 'CATEGORY_EVENT'] = 'Jump'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Relay', na=True), 'CATEGORY_EVENT'] = 'Relay'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Steeple', na=True), 'CATEGORY_EVENT'] = 'Steeple'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Walk', na=True), 'CATEGORY_EVENT'] = 'Walk'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Hurdles', na=True), 'CATEGORY_EVENT'] = 'Hurdles'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Pentathlon', na=True), 'CATEGORY_EVENT'] = 'Pentathlon'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Triathlon', na=True), 'CATEGORY_EVENT'] = 'Triathlon'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Decathlon', na=True), 'CATEGORY_EVENT'] = 'Decathlon'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'Marathon', na=True), 'CATEGORY_EVENT'] = 'Marathon'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'4 x 100m', na=True), 'CATEGORY_EVENT'] = 'Relay'
SGP_athletes.loc[SGP_athletes['EVENT'].str.contains(r'4 x 400m', na=True), 'CATEGORY_EVENT'] = 'Relay'




In [756]:
# Convert to OLD SCHEMA

#SGP_athletes['TEAM'] = 'Singapore'
#SGP_athletes['SEED'] = ''
#SGP_athletes['LANE'] = ''
#SGP_athletes['DIVISION'] = ''
#SGP_athletes['AGE'] = ''
#SGP_athletes['UNIQUE_ID'] = ''
#SGP_athletes['ATHLETE_ID'] = ''
#SGP_athletes['TIMESTAMP'] = ''
#SGP_athletes['TAG_ID'] = ''
#SGP_athletes['POINTS'] = ''
#SGP_athletes['GROUP'] = ''
#SGP_athletes['FREE_FIELD']=''
#SGP_athletes['FREE_FIELD2']=''
#SGP_athletes['FREE_FIELD3']=''


In [ ]:
# Convert to NEW SCHEMA

SGP_athletes['LAST_NAME'] = ''
SGP_athletes['FIRST_NAME'] = ''
SGP_athletes['OTHER_NAME'] = ''
SGP_athletes['TEAM'] = 'Singapore'
SGP_athletes['SEED'] = ''
SGP_athletes['LANE'] = ''
SGP_athletes['DIVISION'] = ''
SGP_athletes['AGE'] = ''
SGP_athletes['UNIQUE_ID'] = ''
SGP_athletes['ATHLETE_ID'] = ''
SGP_athletes['TIMESTAMP'] = ''
SGP_athletes['TAG_ID'] = ''
SGP_athletes['POINTS'] = ''
SGP_athletes['GROUP'] = ''
SGP_athletes['VENUE']=''
SGP_athletes['SUB_EVENT']=''
SGP_athletes['SESSION']=''
SGP_athletes['EVENT_CLASS']=''
SGP_athletes['SUB_EVENT']=''
SGP_athletes['DISTANCE']=''
SGP_athletes['RX_TIME']=''
SGP_athletes['CATEGORY_EVENT']=''


In [757]:
SGP_athletes

,Unnamed: 0,RANK,NAME,NATIONALITY,RESULT,QUALIFICATION,COMPETITION,YEAR,DATE,EVENT,...,AGE,UNIQUE_ID,ATHLETE_ID,TIMESTAMP,TAG_ID,POINTS,GROUP,FREE_FIELD,FREE_FIELD2,FREE_FIELD3
0,102,4.0,Jun Jie Calvin Quek,SGP,53.19,NaN,UAE Athletics Grand Prix,2024,,400m Hurdles,...,,,,,,,,,,
1,31,2.0,Chui Ling Goh,SGP,17:33.73,NaN,11. Münchner Abendsportfest /MUC HochschulMS,2024,,5000m,...,,,,,,,,,,
2,121,4.0,Chen Xiang Ang,SGP,8.01,NaN,11th Asian Indoor Athletics Championships,2024,17 February,60m Hurdles,...,,,,,,,,,,
3,112,8.0,Jun Jie Calvin Quek,SGP,52.05,NaN,11th Kinami Michitaka Memorial Athletics Meet,2024,,400m Hurdles,...,,,,,,,,,,
4,482,4.0,James Ethan Kai Meng Ang,SGP,52.55,NaN,11th Para Athletics World Championships,2024,22 May,400m,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,127,3.0,Jun Jie Calvin Quek,SGP,56.95,NaN,XXXIII Qosanov Memorial,2024,22 June,400m Hurdles,...,,,,,,,,,,
359,148,4.0,Kangli Emery Conrad,SGP,7.22,NaN,XXXIII Qosanov Memorial,2024,22 June,Long Jump,...,,,,,,,,,,
360,320,5.0,Tia Louise Rozario,SGP,5.71,NaN,XXXIII Qosanov Memorial,2024,22 June,Long Jump,...,,,,,,,,,,
361,330,5.0,Tia Louise Rozario,SGP,12.45,NaN,XXXIII Qosanov Memorial,2024,23 June,Triple Jump,...,,,,,,,,,,


In [758]:
df = SGP_athletes.drop(['VENUE', 'YEAR', 'RX_TIME'], axis=1)
df['DATE']='2024'

In [759]:
df

,Unnamed: 0,RANK,NAME,NATIONALITY,RESULT,QUALIFICATION,COMPETITION,DATE,EVENT,GENDER,...,AGE,UNIQUE_ID,ATHLETE_ID,TIMESTAMP,TAG_ID,POINTS,GROUP,FREE_FIELD,FREE_FIELD2,FREE_FIELD3
0,102,4.0,Jun Jie Calvin Quek,SGP,53.19,NaN,UAE Athletics Grand Prix,2024,400m Hurdles,Male,...,,,,,,,,,,
1,31,2.0,Chui Ling Goh,SGP,17:33.73,NaN,11. Münchner Abendsportfest /MUC HochschulMS,2024,5000m,Female,...,,,,,,,,,,
2,121,4.0,Chen Xiang Ang,SGP,8.01,NaN,11th Asian Indoor Athletics Championships,2024,60m Hurdles,Male,...,,,,,,,,,,
3,112,8.0,Jun Jie Calvin Quek,SGP,52.05,NaN,11th Kinami Michitaka Memorial Athletics Meet,2024,400m Hurdles,Male,...,,,,,,,,,,
4,482,4.0,James Ethan Kai Meng Ang,SGP,52.55,NaN,11th Para Athletics World Championships,2024,400m,Male,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,127,3.0,Jun Jie Calvin Quek,SGP,56.95,NaN,XXXIII Qosanov Memorial,2024,400m Hurdles,Male,...,,,,,,,,,,
359,148,4.0,Kangli Emery Conrad,SGP,7.22,NaN,XXXIII Qosanov Memorial,2024,Long Jump,Male,...,,,,,,,,,,
360,320,5.0,Tia Louise Rozario,SGP,5.71,NaN,XXXIII Qosanov Memorial,2024,Long Jump,Female,...,,,,,,,,,,
361,330,5.0,Tia Louise Rozario,SGP,12.45,NaN,XXXIII Qosanov Memorial,2024,Triple Jump,Female,...,,,,,,,,,,


In [760]:
# Re-order columns

df = df.reindex(columns= ['RANK', 'TAG_ID', 'NAME', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION', 'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 
                                      'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'COUNTRY', 'DICT_RESULTS', 'DATE', 'COMPETITION', 'REGION', 'DOB','GROUP', 'CATEGORY_EVENT', 
                                      'ATHLETE_ID', 'SOURCE', 'REMARKS', 'TIMESTAMP', 'FREE_FIELD', 'FREE_FIELD2', 'FREE_FIELD3'])


In [51]:
df

NameError: name 'df' is not defined

In [762]:
df.columns

Index(['RANK', 'TAG_ID', 'NAME', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
       'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE',
       'GENDER', 'UNIQUE_ID', 'COUNTRY', 'DICT_RESULTS', 'DATE', 'COMPETITION',
       'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID', 'SOURCE',
       'REMARKS', 'TIMESTAMP', 'FREE_FIELD', 'FREE_FIELD2', 'FREE_FIELD3'],
      dtype='object')

In [763]:
# Remove special characters
    
for col in df.columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()


    
    

In [764]:
df.reset_index(inplace=True)

In [50]:
# Check file before saving

ct = dt.now()
gender_pattern = 'Male|Female|Mixed'
year_pattern = '2025'
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
df['YEAR'] = df['YEAR'].astype(str)
df['DATE'] = df['DATE'].astype(str)
df['DOB'] = df['DOB'].astype(str)
#df['event_date_dt'] = pd.to_datetime(df['event_date'], format='mixed', dayfirst=False, utc=True)


for index, row in df.iterrows():
        
    assert re.search(gender_pattern, row['GENDER']) is not None   # check format of gender input
    assert row['NAME'] != None
    assert len(row['NAME'])>5
    assert row['DATE'] != None
    assert row['EVENT'] != None 
    assert row['COMPETITION'] != None 
    assert (row['REGION'] == 'Local' or row['REGION'] == 'International')
    assert row['DATE'] is not None
    assert row['CATEGORY_EVENT'] is not None
    assert re.search('\)|\(', row['EVENT']) is None  # make sure there are no brackets in the event column
    assert len(row['DATE'])>3
    assert row['NATIONALITY']=='SGP'
    assert re.search(year_pattern, row['YEAR']) is not None   # check format of year
    
    # Check column locations before loading

    assert len(df.columns) == 40
    assert df.columns.get_loc('RESULT') == 8
    assert df.columns.get_loc('AGE') == 17
    assert df.columns.get_loc('WIND') == 12
    assert df.columns.get_loc('COMPETITION')==24
    
   

    # Set timestamp

    df.loc[:,'TIMESTAMP'] = ct


print('PASSED')

df

NameError: name 'df' is not defined

In [765]:
df.to_csv('SG_athletes_international_2024.csv', encoding='utf-8')

# Process missing data extracted from SAA database

In [237]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/External events/')

athletes = pd.read_csv("ALL_external_data_from_SAA_database_Jan 2011_to_Dec_24.csv")

In [238]:
athletes

,Date,Event,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any"
0,19 Dec 2024,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-
1,19 Dec 2024,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-
2,07 Dec 2024,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-
3,07 Dec 2024,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-
4,07 Dec 2024,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-
...,...,...,...,...,...,...,...,...,...,...
6535,16 Apr 2011,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne"
6536,27 Mar 2011,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee..."
6537,27 Mar 2011,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne"
6538,11 Mar 2011,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand"


In [239]:
# Remove special characters
    
for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()



In [240]:
athletes['REGION'] = 'International' 
athletes['SOURCE'] = 'SAA Database'



In [241]:
# convert Date column into datetime

athletes['Date']=pd.to_datetime(athletes['Date'], format='mixed')

athletes['Date']

0      2024-12-19
1      2024-12-19
2      2024-12-07
3      2024-12-07
4      2024-12-07
          ...    
6535   2011-04-16
6536   2011-03-27
6537   2011-03-27
6538   2011-03-11
6539   2011-02-20
Name: Date, Length: 6540, dtype: datetime64[ns]

In [242]:
# extract day and month only

athletes['date_extract']=athletes['Date'].dt.strftime('%d-%m')
athletes['YEAR']=athletes['Date'].dt.strftime('%Y')

In [243]:
athletes

,Date,Event,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",REGION,SOURCE,date_extract,YEAR
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,International,SAA Database,19-12,2024
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,International,SAA Database,19-12,2024
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,International,SAA Database,07-12,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",International,SAA Database,16-04,2011
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",International,SAA Database,27-03,2011
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",International,SAA Database,27-03,2011
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",International,SAA Database,11-03,2011


In [244]:
mask = athletes['Event'].str.contains(r'Men', na=True)
athletes.loc[mask, 'GENDER'] = 'Male'

mask = athletes['Event'].str.contains(r'Women', na=True)
athletes.loc[mask, 'GENDER'] = 'Female'


In [245]:
athletes

,Date,Event,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",REGION,SOURCE,date_extract,YEAR,GENDER
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,International,SAA Database,19-12,2024,Male
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,International,SAA Database,19-12,2024,Male
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,International,SAA Database,07-12,2024,Male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",International,SAA Database,16-04,2011,Male
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",International,SAA Database,27-03,2011,Female
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",International,SAA Database,27-03,2011,Male
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",International,SAA Database,11-03,2011,Male


In [246]:
athletes['COMPETITION'] = athletes['Competition'].str[16:]  # Remove 'International -' from string

In [247]:
athletes

,Date,Event,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",REGION,SOURCE,date_extract,YEAR,GENDER,COMPETITION
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,International,SAA Database,19-12,2024,Male,Strive (Australia)
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,International,SAA Database,19-12,2024,Male,Vic Milers (Australia)
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)"
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)"
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",International,SAA Database,16-04,2011,Male,Australia Athletics Cships
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",International,SAA Database,27-03,2011,Female,Queensland Open & AWD Cships
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",International,SAA Database,27-03,2011,Male,Athletic Victoria Throwers Meet #6
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",International,SAA Database,11-03,2011,Male,Thai Veterans Athletics Cships


In [248]:
mask = athletes['Info, if any'].str.contains(r'Final', na=True)
athletes.loc[mask, 'STAGE'] = 'Final'

mask = athletes['Info, if any'].str.contains(r'Prelim', na=True)
athletes.loc[mask, 'STAGE'] = 'Prelim'

mask = athletes['Info, if any'].str.contains(r'Semifinal', na=True)
athletes.loc[mask, 'STAGE'] = 'Semi'

mask = athletes['Info, if any'].str.contains(r'Semi', na=True)
athletes.loc[mask, 'STAGE'] = 'Semi'


mask = athletes['Info, if any'].str.contains(r'Heats', na=True)
athletes.loc[mask, 'STAGE'] = 'Heats'


In [249]:
mask = athletes['Info, if any'].str.contains(r'U20', na=True)
athletes.loc[mask, 'DIVISION'] = 'U20'

mask = athletes['Info, if any'].str.contains(r'U18', na=True)
athletes.loc[mask, 'DIVISION'] = 'U18'


In [250]:
'''

mask = athletes['Info, if any'].str.contains(r'PB', na=True)
athletes.loc[mask, 'REMARKS'] = 'PB'

mask = athletes['Info, if any'].str.contains(r'SB', na=True)
athletes.loc[mask, 'REMARKS'] = 'SB'

mask = athletes['Info, if any'].str.contains(r'CR', na=True)
athletes.loc[mask, 'REMARKS'] = 'CR'

mask = athletes['Info, if any'].str.contains(r'NR', na=True)
athletes.loc[mask, 'REMARKS'] = 'NR'

mask = athletes['Info, if any'].str.contains(r'R1', na=True)
athletes.loc[mask, 'REMARKS'] = 'R1'

'''


"\n\nmask = athletes['Info, if any'].str.contains(r'PB', na=True)\nathletes.loc[mask, 'REMARKS'] = 'PB'\n\nmask = athletes['Info, if any'].str.contains(r'SB', na=True)\nathletes.loc[mask, 'REMARKS'] = 'SB'\n\nmask = athletes['Info, if any'].str.contains(r'CR', na=True)\nathletes.loc[mask, 'REMARKS'] = 'CR'\n\nmask = athletes['Info, if any'].str.contains(r'NR', na=True)\nathletes.loc[mask, 'REMARKS'] = 'NR'\n\nmask = athletes['Info, if any'].str.contains(r'R1', na=True)\nathletes.loc[mask, 'REMARKS'] = 'R1'\n\n"

In [251]:
athletes.rename(columns={'Event': 'EVENT'}, inplace=True)


In [252]:
# Running

mask = athletes['EVENT'].str.contains(r'50 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '50m'
mask = athletes['EVENT'].str.contains(r'60 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '60m'
mask = athletes['EVENT'].str.contains(r'80 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '80m'
mask = athletes['EVENT'].str.contains(r'100 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'100 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'^100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'200 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = athletes['EVENT'].str.contains(r'^200m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = athletes['EVENT'].str.contains(r'200\sMeter', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = athletes['EVENT'].str.contains(r'300 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '300m'
mask = athletes['EVENT'].str.contains(r'400 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = athletes['EVENT'].str.contains(r'^400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'

mask = athletes['EVENT'].str.contains(r'^400\sMeter$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'


mask = athletes['EVENT'].str.contains(r'600 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '600m'
mask = athletes['EVENT'].str.contains(r'800 Meter Dash', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'800 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'^800m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'1500 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = athletes['EVENT'].str.contains(r'^1500m$', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = athletes['EVENT'].str.contains(r'3000 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
mask = athletes['EVENT'].str.contains(r'3000m', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
mask = athletes['EVENT'].str.contains(r'5000 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'
mask = athletes['EVENT'].str.contains(r'5000m', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'
mask = athletes['EVENT'].str.contains(r'10000 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10000m'
mask = athletes['EVENT'].str.contains(r'^10000m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'
mask = athletes['EVENT'].str.contains(r'10 Kilometer Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'


mask = athletes['EVENT'].str.contains(r'1 Mile Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1 mile'
mask = athletes['EVENT'].str.contains(r'Mile', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1 mile'


mask = athletes['EVENT'].str.contains(r'10\,000m', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'



# Hurdles

mask = athletes['EVENT'].str.contains(r'^80m\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '80m Hurdles'
mask = athletes['EVENT'].str.contains(r'^80m\shurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '80m Hurdles'
mask = athletes['EVENT'].str.contains(r'^80\sMeter\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '80m Hurdles'
mask = athletes['EVENT'].str.contains(r'^100m\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = athletes['EVENT'].str.contains(r'^100\sMeter\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m hurdles'
mask = athletes['EVENT'].str.contains(r'100\sMeter\sHurdles\s\(0\.838m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = athletes['EVENT'].str.contains(r'100m\sHurdles\s\(0\.838m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'

mask = athletes['EVENT'].str.contains(r'110\sMeter\sHurdles', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = athletes['EVENT'].str.contains(r'^110m\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'


#mask = athletes['EVENT'].str.contains(r'110\sMeter\sHurdles\s\(0\.914m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m hurdles'
#mask = athletes['EVENT'].str.contains(r'110m\sHurdles\s\(0\.914m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m hurdles'

mask = athletes['EVENT'].str.contains(r'110\sMeter\sHurdles\s\(1\.067m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = athletes['EVENT'].str.contains(r'110m\sHurdles\s\(1\.067m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = athletes['EVENT'].str.contains(r'110\sMeter\sHurdles\sOpen', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'



mask = athletes['EVENT'].str.contains(r'^200m\sHurdles$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m Hurdles'
mask = athletes['EVENT'].str.contains(r'200 Meter Hurdles', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m Hurdles'
mask = athletes['EVENT'].str.contains(r'200m Hurdles', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m Hurdles'
mask = athletes['EVENT'].str.contains(r'200m\sHurdles\s\(0\.762m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = ' '

mask = athletes['EVENT'].str.contains(r'400m\sHurdles', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = athletes['EVENT'].str.contains(r'400m\sHurdles\s\(0.840m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = ' '

mask = athletes['EVENT'].str.contains(r'400\sMeter\sHurdles', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'


mask = athletes['EVENT'].str.contains(r'400m\sHurdles\sOpen', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = athletes['EVENT'].str.contains(r'400\sMeter\sHurdles\sOpen', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'

mask = athletes['EVENT'].str.contains(r'400\sMeter\sHurdles\sOPEN', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'


mask = athletes['EVENT'].str.contains(r'400\sMeter\sHurdles\s\(0\.914m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m hurdles'
mask = athletes['EVENT'].str.contains(r'400m\sHurdles\s\(0\.914m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'

mask = athletes['EVENT'].str.contains(r'^400\sMeter\sHurdles\s\(0\.762m\)$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = athletes['EVENT'].str.contains(r'^400m\sHurdles\s\(0\.762m\)$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'


# Throws

mask = athletes['EVENT'].str.contains(r'Javelin\sThrow\s\(600g\)', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = athletes['EVENT'].str.contains(r'^Javelin\sThrow$', na=True, regex=True)  # exact match for string
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s\(600g\)', na=True, regex=True)) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s600g', na=True, regex=True)) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s600g\)', na=True, regex=True)) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin throw'
mask = athletes['EVENT'].str.contains(r'Javelin\sThrow\s\(800g\)', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = athletes['EVENT'].str.contains(r'Javelin\sThrow\sOpen', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = athletes['EVENT'].str.contains(r'Javelin\sThrow\sOPEN', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = athletes['EVENT'].str.contains(r'Javelin\sThrow', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'



mask = athletes['EVENT'].str.contains(r'^Shot\sPut$', na=True, regex=True) # there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

mask = (athletes['EVENT'].str.contains(r'Shot\sPut\s\(4\.00kg\)', na=True, regex=True) & (athletes['GENDER']=='Female'))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = (athletes['EVENT'].str.contains(r'Shot\sPut\s\(4kg\)', na=True, regex=True) & (athletes['GENDER']=='Female'))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'


mask = athletes['EVENT'].str.contains(r'Women\sShot\sPut\s4kg\sOpen', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = athletes['EVENT'].str.contains(r'Men\sShot\sPut\s4kg\sOPEN', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot put'
mask = athletes['EVENT'].str.contains(r'Shot\sPut\sOPEN', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'


mask = athletes['EVENT'].str.contains(r'Women\sShot\sPut\s\(4kg\)', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = athletes['EVENT'].str.contains(r'Shot\sPut\s\(7\.26kg\)', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = athletes['EVENT'].str.contains(r'Shot\sPut\s7\.26kg\sOpen', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

mask = athletes['EVENT'].str.contains(r'Shot\sPut\sOpen', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = athletes['EVENT'].str.contains(r'^Shot\sput$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'


#mask = athletes['EVENT'].str.contains(r'Shot Put Masters', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot put'


mask = athletes['EVENT'].str.contains(r'^Hammer\sThrow$', na=True)  # there are some characters after Throw
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'

mask = (athletes['EVENT'].str.contains(r'Hammer\sThrow\s\(4kg\)', na=True) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
mask = athletes['EVENT'].str.contains(r'Hammer\sThrow\s\(7\.26kg\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'

#mask = athletes['EVENT'].str.contains(r'^Discus\sThrow    $', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus throw'
mask = athletes['EVENT'].str.contains(r'Discus\sThrow', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = athletes['EVENT'].str.contains(r'Discus\sthrow', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'


mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=True)) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

mask = ((athletes['EVENT'].str.contains(r'Discus\s\(1\.00kg\)', na=True))  & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'



mask = athletes['EVENT'].str.contains(r'Women\sDiscus\sThrow\s\(1kg\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'



mask = athletes['EVENT'].str.contains(r'Discus\sThrow\s\(2kg\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=True)) & (athletes['GENDER']=='Female'))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = athletes['EVENT'].str.contains(r'Discus\sThrow\sOpen', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = athletes['EVENT'].str.contains(r'Discus\sThrow\sOPEN', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'



# Jumps

mask = athletes['EVENT'].str.contains(r'High Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'

mask = athletes['EVENT'].str.contains(r'Long\sJump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Open', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Trial', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'


mask = athletes['EVENT'].str.contains(r'Triple Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'Pole Vault', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'
mask = athletes['EVENT'].str.contains(r'High jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'
mask = athletes['EVENT'].str.contains(r'Long jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Triple jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'^Pole\svault$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'

# Steeplechase

mask = athletes['EVENT'].str.contains(r'2000m S/C', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = athletes['EVENT'].str.contains(r'2000m Steeplechase', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = athletes['EVENT'].str.contains(r'2000 Meter Steeplechase', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = athletes['EVENT'].str.contains(r'3000m S/C', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Steeplechase'
mask = athletes['EVENT'].str.contains(r'3000 Meter Steeplechase', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Steeplechase'

# Marathon

mask = athletes['EVENT'].str.contains(r'Marathon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Marathon'
mask = athletes['EVENT'].str.contains(r'Half\sMarathon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Half marathon'

# Walk

mask = athletes['EVENT'].str.contains(r'1500m Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m Racewalk'
mask = athletes['EVENT'].str.contains(r'1500 Meter Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m Racewalk'
mask = athletes['EVENT'].str.contains(r'3000m Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Racewalk'
mask = athletes['EVENT'].str.contains(r'3000 Meter Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Racewalk'
mask = athletes['EVENT'].str.contains(r'5000 Meter Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m Racewalk'
mask = athletes['EVENT'].str.contains(r'5000m Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m Racewalk'
mask = athletes['EVENT'].str.contains(r'10000 Meter Race Walk', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m Racewalk'

# Relay

mask = athletes['EVENT'].str.contains(r'4x80m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 80m Relay'
mask = athletes['EVENT'].str.contains(r'^4\sx\s100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m Relay'
mask = athletes['EVENT'].str.contains(r'4x100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m Relay'
mask = athletes['EVENT'].str.contains(r'4 X 100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m Relay'
mask = athletes['EVENT'].str.contains(r'4x400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m Relay'
mask = athletes['EVENT'].str.contains(r'4 X 400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m Relay'
mask = athletes['EVENT'].str.contains(r'4x100 Meter Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m Relay'
mask = athletes['EVENT'].str.contains(r'4x400 Meter Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m Relay'
mask = athletes['EVENT'].str.contains(r'^4\sx\s400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m Relay'

# Decathlon/Heptathlon

mask = athletes['EVENT'].str.contains(r'^Heptathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'^Decathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Decathlon'
mask = athletes['EVENT'].str.contains(r'Heptathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'Decathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Decathlon'

# Cross Country

mask = athletes['EVENT'].str.contains(r'10 Kilometer Run (Cross Country)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10km Cross Country'



/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_13049/745616938.py:340: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = athletes['EVENT'].str.contains(r'10 Kilometer Run (Cross Country)', na=True)


In [253]:
athletes

,Date,EVENT,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",REGION,SOURCE,date_extract,YEAR,GENDER,COMPETITION,STAGE,DIVISION,MAPPED_EVENT
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,International,SAA Database,19-12,2024,Male,Strive (Australia),NaN,NaN,800m
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,International,SAA Database,19-12,2024,Male,Vic Milers (Australia),NaN,NaN,1500m
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,200m
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,International,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",International,SAA Database,16-04,2011,Male,Australia Athletics Cships,Prelim,NaN,Shot Put
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",International,SAA Database,27-03,2011,Female,Queensland Open & AWD Cships,Final,NaN,10000m
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",International,SAA Database,27-03,2011,Male,Athletic Victoria Throwers Meet #6,Final,NaN,Shot Put
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",International,SAA Database,11-03,2011,Male,Thai Veterans Athletics Cships,Final,NaN,5000m Racewalk


In [254]:
# Extract ranking

def rank(string):
    
    

    try:
        
        slash = re.search(r'\/', string) 
        
        d1 = re.search(r'^\dpos', string)
        d2 = re.search(r'^\d\dpos', string)
        d3 = re.search(r'^\d\d\d\dpos', string)
        d4 = re.search(r'^\d\d\dpos', string)
        
        print(d1, d2, d3, d4)
        
        print(slash)
        
        if slash:
     
            start = slash.start()-2
    
            end = slash.start()
    
            pos = string[start:end]
        
        elif d1:
            
            pos = string[:1]
            
            print(d1)
            
        elif d2:
            
            pos = string[:2]
            
            print(d2)
            
        elif d3:
            
            pos = string[:4]
            
        elif d4:
            
            pos = string[:3]
            
            
        else:
            
            pos=''
            
    except:
        
        pos = ''
    
    return pos
    

athletes['RANK'] = athletes['Info, if any'].apply(rank)


None None None None
None
None None None None
None
None None None None
None
None None None None
None
None None None None
None
None None None None
None
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(12, 13), match='/'>
None None None None
<re.Match object; span=(12, 13), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(12, 13), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; span=(8, 9), match='/'>
None None None None
<re.Match object; 

In [6]:
# Extract date

def date(string):
    
    
    try:
        
        slash=re.search(r'2025', string)
        
        end = slash.start()
    
        date = string[:end]
            
    except:
        
        date = ''
    
    return date
    

#athletes['DATE'] = athletes['Date'].apply(date)


In [7]:
athletes

NameError: name 'athletes' is not defined

In [257]:
# Extract heat

def heat(string):
    
    
    try:
        
        heat=re.findall(r'h\d|H\d|h\d\d|H\d\d', string)
        
        heat1=heat[0]
                
            
    except:
        
        heat1 = ''
    
    return heat1
    

athletes['HEAT'] = athletes['Info, if any'].apply(heat)


In [258]:
athletes

,Date,EVENT,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",...,SOURCE,date_extract,YEAR,GENDER,COMPETITION,STAGE,DIVISION,MAPPED_EVENT,RANK,HEAT
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,...,SAA Database,19-12,2024,Male,Strive (Australia),NaN,NaN,800m,,
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,...,SAA Database,19-12,2024,Male,Vic Milers (Australia),NaN,NaN,1500m,,
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,...,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m,,
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,...,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,200m,,
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,...,SAA Database,07-12,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",...,SAA Database,16-04,2011,Male,Australia Athletics Cships,Prelim,NaN,Shot Put,,
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",...,SAA Database,27-03,2011,Female,Queensland Open & AWD Cships,Final,NaN,10000m,,
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",...,SAA Database,27-03,2011,Male,Athletic Victoria Throwers Meet #6,Final,NaN,Shot Put,,
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",...,SAA Database,11-03,2011,Male,Thai Veterans Athletics Cships,Final,NaN,5000m Racewalk,,


In [5]:
# Extract qualitication

def qual(string):
    
    
    try:
        
        qual=re.findall(r'\sq|\sQ', string)
        qual1=qual[0]
        
                
            
    except:
        
        qual1 = ''
    
    return qual1
    

athletes['QUALIFICATION'] = athletes['Info, if any'].apply(qual)


NameError: name 'athletes' is not defined

In [260]:
# Extract event class

def event_class(string):
    
    
    try:
        
        left=re.search(r'\(', string)
        right=re.search(r'\)', string)
        
        
        type = string[left.start()+1:right.start()]
        
                 
    except:
        
        type = ''
    
    return type
    

athletes['EVENT_CLASS'] = athletes['EVENT'].apply(event_class)


In [261]:
athletes

,Date,EVENT,Name,Age,Team,Result,Wind m/s,Competition,Year D.O.B.,"Info, if any",...,YEAR,GENDER,COMPETITION,STAGE,DIVISION,MAPPED_EVENT,RANK,HEAT,QUALIFICATION,EVENT_CLASS
0,2024-12-19,Men 800 Meter Run,"THANA RAJAN, THIRUBEN",24,SINGAPORE,1:53.51,-,International - Strive (Australia),2000,-,...,2024,Male,Strive (Australia),NaN,NaN,800m,,,,
1,2024-12-19,Men 1500 Meter Run,"LIM, OLIVER",24,SINGAPORE,4:01.49,-,International - Vic Milers (Australia),1999,-,...,2024,Male,Vic Milers (Australia),NaN,NaN,1500m,,,,
2,2024-12-07,Men 60 Meter Dash,"TAN, DARYL",23,SINGAPORE,6.84,1.4,"International - Strive Program B, Perth (Austr...",2001,-,...,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m,,,,
3,2024-12-07,Men 200 Meter Dash,"TAN, DARYL",23,SINGAPORE,21.68,1.2,"International - Strive Program B, Perth (Austr...",2001,-,...,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,200m,,,,
4,2024-12-07,Men 60 Meter Dash,"TOH JUN XI, TEDD",22,SINGAPORE,6.97,1.3,"International - Strive Program B, Perth (Austr...",2002,-,...,2024,Male,"Strive Program B, Perth (Australia)",NaN,NaN,60m,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,2011-04-16,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,14.91,-,International - Australia Athletics Cships,1990,"- Preliminaries Olympic Park, Melbourne",...,2011,Male,Australia Athletics Cships,Prelim,NaN,Shot Put,,,,7.26kg
6536,2011-03-27,Women 10000 Meter Run,Renuka Satianathan,24,Singapore,37:15.87,-,International - Queensland Open & AWD Cships,1987,"- Finals State Athletics Facility, Nathan Quee...",...,2011,Female,Queensland Open & AWD Cships,Final,NaN,10000m,,,Q,
6537,2011-03-27,Men Shot Put (7.26kg),Scott Wong Wei Gen,21,Singapore,15.32,-,International - Athletic Victoria Throwers Mee...,1990,"- Finals Proclamations Park, Ringwood, Melbourne",...,2011,Male,Athletic Victoria Throwers Meet #6,Final,NaN,Shot Put,,,,7.26kg
6538,2011-03-11,Men 5000 Meter Race Walk,Peter James Back,-,Singapore,26:53,HT,International - Thai Veterans Athletics Cships,-,"Open Final Bangkok, Thailand",...,2011,Male,Thai Veterans Athletics Cships,Final,NaN,5000m Racewalk,,,,


In [262]:
athletes.to_csv("SAA_external_database.csv", encoding="utf-8")

In [263]:
athletes = athletes.rename(columns={'Info, if any': 'REMARKS', 'Wind m/s': "WIND", 'Age': "AGE", 'Result': "RESULT", 'date_extract': "DATE", 'Year D.O.B.': "DOB"})

In [264]:
athletes = athletes.drop(['Date', 'Competition', 'EVENT'], axis=1)

In [265]:
athletes = athletes.rename(columns={'MAPPED_EVENT': "EVENT", 'Name': "NAME", 'Team': "TEAM"})

In [266]:
athletes = athletes[athletes['YEAR']=='2023']

In [267]:
athletes

,NAME,AGE,TEAM,RESULT,WIND,DOB,REMARKS,REGION,SOURCE,DATE,YEAR,GENDER,COMPETITION,STAGE,DIVISION,EVENT,RANK,HEAT,QUALIFICATION,EVENT_CLASS
530,"NORHISHAM, JAMIE EL-REDHA ANG",19,Singapore Sports School,11.01,-0.2,2004,Heats 1/8pos q,International,SAA Database,20-12,2023,Male,SAECA-BJSS Athletics Championship,Heats,NaN,100m,1,,q,
531,"NORHISHAM, JAMIE EL-REDHA ANG",19,Singapore Sports School,11.08,-0.1,2004,Final 8/8pos,International,SAA Database,20-12,2023,Male,SAECA-BJSS Athletics Championship,Final,NaN,100m,8,,,
532,"MUTHUKUMARAN, RAAM KUMAR",19,Singapore Sports School,3:20.75,-,2004,Final 1/8pos,International,SAA Database,20-12,2023,Male,SAECA-BJSS Athletics Championship,Final,NaN,4 x 400m Relay,1,,,
533,"NORHISHAM, JAMIE EL-REDHA ANG",19,Singapore Sports School,22.32,-0.3,2004,Final 3/8pos,International,SAA Database,19-12,2023,Male,SAECA-BJSS Athletics Championship,Final,NaN,200m,3,,,
534,"YUSRAN, NIQ RIFDI RAFIQI",16,Singapore Sports School,55.05,-,2007,Heats 4/8pos,International,SAA Database,19-12,2023,Male,SAECA-BJSS Athletics Championship,Heats,NaN,400m,4,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2114,"ROZARIO, TIA LOUISE",23,PRINCETON,5.52,Indoor,2000,"5/8pos, sanctioned",International,SAA Database,28-01,2023,Female,Harvard-Yale-Princeton,NaN,NaN,Long Jump,,,,
2115,"KAM, KAMPTON",22,INDIVIDUAL,2.08,-,2001,"Sanctioned,1pos",International,SAA Database,21-01,2023,Male,Wesley A Brown Invitational,NaN,NaN,High Jump,,,,
2116,"PHUA, JASMIN",22,Individual,42.83,-,2000,-,International,SAA Database,20-01,2023,Female,Strive Season Program B,NaN,NaN,Discus Throw,,,,1kg
2117,"KAM, KAMPTON",22,Individual,2.06,-,2000,"Sanctioned,1pos",International,SAA Database,14-01,2023,Male,Penn 10-Team Select,NaN,NaN,High Jump,,,,


In [268]:
athletes.to_csv("athletes_2023.csv", encoding='utf-8')

In [269]:
# Map general event category 

athletes.loc[athletes['EVENT'].str.contains(r'100m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'400m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'600m', na=True), 'CATEGORY_EVENT'] = 'Mid'
athletes.loc[athletes['EVENT'].str.contains(r'60m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'200m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'50m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'80m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'300m', na=True), 'CATEGORY_EVENT'] = 'Sprint'
athletes.loc[athletes['EVENT'].str.contains(r'3000m', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'Mile', na=True), 'CATEGORY_EVENT'] = 'Mid'
athletes.loc[athletes['EVENT'].str.contains(r'mile', na=True), 'CATEGORY_EVENT'] = 'Mid'

athletes.loc[athletes['EVENT'].str.contains(r'5000m', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'2000m', na=True), 'CATEGORY_EVENT'] = 'Mid'
athletes.loc[athletes['EVENT'].str.contains(r'800m', na=True), 'CATEGORY_EVENT'] = 'Mid'
athletes.loc[athletes['EVENT'].str.contains(r'1500m', na=True), 'CATEGORY_EVENT'] = 'Mid'
athletes.loc[athletes['EVENT'].str.contains(r'10,000m', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'10000m', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'5km', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'10km', na=True), 'CATEGORY_EVENT'] = 'Long'
athletes.loc[athletes['EVENT'].str.contains(r'Cross Country', na=True), 'CATEGORY_EVENT'] = 'Cross Country'


# Override values

athletes.loc[athletes['EVENT'].str.contains(r'Discus', na=True), 'CATEGORY_EVENT'] = 'Throw'
athletes.loc[athletes['EVENT'].str.contains(r'Javelin', na=True), 'CATEGORY_EVENT'] = 'Throw'
athletes.loc[athletes['EVENT'].str.contains(r'Shot', na=True), 'CATEGORY_EVENT'] = 'Throw'
athletes.loc[athletes['EVENT'].str.contains(r'Vault', na=True), 'CATEGORY_EVENT'] = 'Jump'
athletes.loc[athletes['EVENT'].str.contains(r'vault', na=True), 'CATEGORY_EVENT'] = 'Jump'

athletes.loc[athletes['EVENT'].str.contains(r'Throw', na=True), 'CATEGORY_EVENT'] = 'Throw'
athletes.loc[athletes['EVENT'].str.contains(r'Jump', na=True), 'CATEGORY_EVENT'] = 'Jump'
athletes.loc[athletes['EVENT'].str.contains(r'jump', na=True), 'CATEGORY_EVENT'] = 'Jump'

athletes.loc[athletes['EVENT'].str.contains(r'Relay', na=True), 'CATEGORY_EVENT'] = 'Relay'
athletes.loc[athletes['EVENT'].str.contains(r'steeple', na=True), 'CATEGORY_EVENT'] = 'Steeple'
athletes.loc[athletes['EVENT'].str.contains(r'Walk', na=True), 'CATEGORY_EVENT'] = 'Walk'
athletes.loc[athletes['EVENT'].str.contains(r'Hurdles', na=True), 'CATEGORY_EVENT'] = 'Hurdles'
athletes.loc[athletes['EVENT'].str.contains(r'hurdles', na=True), 'CATEGORY_EVENT'] = 'Hurdles'

athletes.loc[athletes['EVENT'].str.contains(r'Pentathlon', na=True), 'CATEGORY_EVENT'] = 'Pentathlon'
athletes.loc[athletes['EVENT'].str.contains(r'Triathlon', na=True), 'CATEGORY_EVENT'] = 'Triathlon'
athletes.loc[athletes['EVENT'].str.contains(r'Decathlon', na=True), 'CATEGORY_EVENT'] = 'Decathlon'
athletes.loc[athletes['EVENT'].str.contains(r'Marathon', na=True), 'CATEGORY_EVENT'] = 'Marathon'
athletes.loc[athletes['EVENT'].str.contains(r'marathon', na=True), 'CATEGORY_EVENT'] = 'Marathon'
athletes.loc[athletes['EVENT'].str.contains(r'4 x 100m', na=True), 'CATEGORY_EVENT'] = 'Relay'
athletes.loc[athletes['EVENT'].str.contains(r'4 x 400m', na=True), 'CATEGORY_EVENT'] = 'Relay'
athletes.loc[athletes['EVENT'].str.contains(r'Steeplechase', na=True), 'CATEGORY_EVENT'] = 'Steeple'




In [270]:
# Extract date

def time(string):
    
    
    try:
        
        slash=re.search(r'AM|PM', string)
        
        end = slash.start()
    
        time = string[:end]
            
    except:
        
        time = ''
    
    return time
    

#athletes['RESULT_CLEANED'] = athletes['RESULT'].apply(time)


In [271]:
athletes.columns

Index(['NAME', 'AGE', 'TEAM', 'RESULT', 'WIND', 'DOB', 'REMARKS', 'REGION',
       'SOURCE', 'DATE', 'YEAR', 'GENDER', 'COMPETITION', 'STAGE', 'DIVISION',
       'EVENT', 'RANK', 'HEAT', 'QUALIFICATION', 'EVENT_CLASS',
       'CATEGORY_EVENT'],
      dtype='object')

In [272]:
# Convert to NEW SCHEMA

athletes['LAST_NAME'] = ''
athletes['FIRST_NAME'] = ''
athletes['OTHER_NAME'] = ''
athletes['SEED'] = ''
athletes['LANE'] = ''
athletes['ATHLETE_ID'] = ''
athletes['TIMESTAMP'] = ''
athletes['TAG_ID'] = ''
athletes['POINTS'] = ''
athletes['GROUP'] = ''
athletes['SESSION']=''
athletes['DISTANCE']=''
athletes['HOST_CITY']=''
athletes['SUB_EVENT']=''
athletes['DICT_RESULTS']=''
athletes['RX_TIME']=''






athletes = athletes.reindex(columns= ['FIRST_NAME', 'LAST_NAME', 'OTHER_NAME', 'NAME', 'RANK', 'TAG_ID', 'TEAM', 'SEED', 'RESULT', 'QUALIFICATION',
                                        'HEAT', 'LANE', 'WIND', 'EVENT', 'DIVISION', 'STAGE', 'POINTS', 'AGE', 'GENDER', 'UNIQUE_ID', 'NATIONALITY',
                                        'DICT_RESULTS', 'YEAR', 'DATE', 'COMPETITION', 'REGION', 'DOB', 'GROUP', 'CATEGORY_EVENT', 'ATHLETE_ID',
                                        'SOURCE', 'REMARKS', 'TIMESTAMP', 'VENUE', 'SUB_EVENT', 'SESSION', 'EVENT_CLASS', 'DISTANCE', 'HOST_CITY', 'RX_TIME'])


In [273]:
Marathon_2023 = athletes[athletes['EVENT']=='Marathon']

In [275]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SAA Database/')


Marathon_2023.to_csv("SAA_database_marathon_2023.csv", encoding='utf-8')

In [56]:
athletes

,FIRST_NAME,LAST_NAME,OTHER_NAME,NAME,RANK,TAG_ID,TEAM,SEED,RESULT,QUALIFICATION,...,SOURCE,REMARKS,TIMESTAMP,VENUE,SUB_EVENT,SESSION,EVENT_CLASS,DISTANCE,HOST_CITY,RX_TIME
0,,,,NaN,,,NaN,,1:53.51,,...,SAA Database,NaN,,NaN,,,,,,
1,,,,NaN,,,NaN,,4:01.49,,...,SAA Database,NaN,,NaN,,,,,,
2,,,,NaN,,,NaN,,6.84,,...,SAA Database,NaN,,NaN,,,,,,
3,,,,NaN,,,NaN,,21.68,,...,SAA Database,NaN,,NaN,,,,,,
4,,,,NaN,,,NaN,,6.97,,...,SAA Database,NaN,,NaN,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6535,,,,NaN,,,NaN,,14.91,,...,SAA Database,NaN,,NaN,,,7.26kg,,,
6536,,,,NaN,,,NaN,,37:15.87,Q,...,SAA Database,NaN,,NaN,,,,,,
6537,,,,NaN,,,NaN,,15.32,,...,SAA Database,NaN,,NaN,,,7.26kg,,,
6538,,,,NaN,,,NaN,,26:53,,...,SAA Database,NaN,,NaN,,,,,,


# Check googe sheets downloand and retrieval

In [122]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Google Sheets/2025/Aug/')

google = pd.read_csv("SAA Competitions - Competitions.csv")


In [123]:
google

,Competition Name,Country,Competition Date,Athlete Name,Event,Result,Wind,Gender,Comments,Uploads
0,"Queensland 10,000m Championships",Australia,09/08/2025,Shaun Goh,10000m,31:02.40,NaN,Male,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/196cOw1tYQUkRY...
1,"Queensland 10,000m Championships",Australia,09/08/2025,Vanessa Lee,10000m,36:15.67,NaN,Female,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/196cOw1tYQUkRY...
2,7th Tokai Sprint Games,Japan,17/08/2025,Calvin Quek,100m,10.72,0.4,Male,https://worldathletics.org/competition/calenda...,NaN
3,7th Tokai Sprint Games,Japan,17/08/2025,Praharsh Ryan,100m,10.70,0.0,Male,https://worldathletics.org/competition/calenda...,NaN
4,7th Tokai Sprint Games,Japan,17/08/2025,Praharsh Ryan,100m,10.70,0.1,Male,https://worldathletics.org/competition/calenda...,NaN
5,2025 ICAAK Games,Japan,12/08/2025,Calvin Quek,400m Hurdles (0.914m),50.24,NaN,Male,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/13TsWlLosDFnOu...
6,20th Twilight Games,Japan,20/08/2025,Calvin Quek,400m Hurdles (0.914m),49.75,NaN,Male,NaN,NaN


In [125]:
google['Result']=google['Result'].astype(str)

In [126]:
google

,Competition Name,Country,Competition Date,Athlete Name,Event,Result,Wind,Gender,Comments,Uploads
0,"Queensland 10,000m Championships",Australia,09/08/2025,Shaun Goh,10000m,31:02.40,NaN,Male,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/196cOw1tYQUkRY...
1,"Queensland 10,000m Championships",Australia,09/08/2025,Vanessa Lee,10000m,36:15.67,NaN,Female,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/196cOw1tYQUkRY...
2,7th Tokai Sprint Games,Japan,17/08/2025,Calvin Quek,100m,10.72,0.4,Male,https://worldathletics.org/competition/calenda...,NaN
3,7th Tokai Sprint Games,Japan,17/08/2025,Praharsh Ryan,100m,10.70,0.0,Male,https://worldathletics.org/competition/calenda...,NaN
4,7th Tokai Sprint Games,Japan,17/08/2025,Praharsh Ryan,100m,10.70,0.1,Male,https://worldathletics.org/competition/calenda...,NaN
5,2025 ICAAK Games,Japan,12/08/2025,Calvin Quek,400m Hurdles (0.914m),50.24,NaN,Male,https://worldathletics.org/competition/calenda...,https://drive.google.com/file/d/13TsWlLosDFnOu...
6,20th Twilight Games,Japan,20/08/2025,Calvin Quek,400m Hurdles (0.914m),49.75,NaN,Male,NaN,NaN
